# ApexID CVPR Stage 01
## BHCA backbone extraction and validation-only rejection calibration

This notebook extracts frozen L2 embeddings from **three BHCA ConvNeXt-Small RGB backbones**, then fits the official `margin_gallery_logistic` rejector on validation only.

The three checkpoints replace `ApexID_HA_v2`, `ApexID_v2`, MiewIDv3, and MegaDescriptor. Each Kaggle model folder stores `best_backbone.pt` next to its sidecar config/metadata (`config_resolved.yaml`, `best_metadata.json`). Those sidecars are the only source of architecture and preprocessing; nothing is read from a nested `checkpoints/` directory.

Extraction writes Stage-02-compatible `.npz` caches. Calibration still uses only `train_gallery`, `val_known`, and identity-disjoint `val_unknown`. No final-test label enters episode construction, feature standardization, logistic fitting, hyperparameters, or official threshold selection.


In [ ]:
# ============================================================
# 1. Local configuration — three BHCA RGB backbones
# ============================================================

import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()


GITHUB_REPO = os.environ.get("APEXID_GIT_REPO", "https://github.com/MIA-AI-Team/ApexID.git")
GITHUB_BRANCH = os.environ.get("APEXID_GIT_BRANCH", "pretrained_training_cluster")


def _is_apexid_repo(path: Path) -> bool:
    return (path / "apexid" / "__init__.py").is_file()


def _iter_search_roots():
    here = Path.cwd().resolve()
    yield here
    yield here.parent
    yield Path("/kaggle/working")
    yield Path("/kaggle/working/ApexID")
    yield Path("/kaggle/input")
    yield Path(r"C:\Users\Legion 5\Desktop\ApexID")
    yield from here.parents


def _find_existing_repo() -> Path | None:
    seen = set()
    for candidate in _iter_search_roots():
        if not candidate.exists():
            continue
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if _is_apexid_repo(resolved):
            return resolved
        if resolved.is_dir():
            nested = resolved / "ApexID"
            if _is_apexid_repo(nested):
                return nested
            try:
                children = list(resolved.iterdir())
            except OSError:
                continue
            for child in children:
                if not child.is_dir():
                    continue
                if _is_apexid_repo(child):
                    return child
                deeper = child / "ApexID"
                if _is_apexid_repo(deeper):
                    return deeper
    return None


def _kaggle_secret(*names: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
    except Exception:
        return None
    for name in names:
        try:
            token = secrets.get_secret(name)
        except Exception:
            token = None
        if token:
            return token
    return None


def _github_token() -> str | None:
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("github_token")
    if token:
        return token
    return _kaggle_secret("GITHUB_TOKEN", "github_token")


def _hf_token() -> str | None:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    token = _kaggle_secret("hugging_face", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")
    if token:
        os.environ.setdefault("HF_TOKEN", token)
        os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", token)
    return token


def _clone_url(token: str | None) -> str:
    if token and GITHUB_REPO.startswith("https://github.com/"):
        return GITHUB_REPO.replace("https://github.com/", f"https://{token}@github.com/", 1)
    return GITHUB_REPO


def _ensure_repo_root() -> Path:
    existing = _find_existing_repo()
    if existing is not None:
        return existing
    destination = Path("/kaggle/working/ApexID") if ON_KAGGLE else (Path.cwd().resolve() / "ApexID")
    if _is_apexid_repo(destination):
        return destination
    token = _github_token()
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"ApexID package not found; cloning {GITHUB_REPO} ({GITHUB_BRANCH}) into {destination}")
    command = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, _clone_url(token), str(destination)]
    try:
        subprocess.check_call(command)
    except subprocess.CalledProcessError as exc:
        raise FileNotFoundError(
            "Could not locate or clone the ApexID repository (apexid/__init__.py). "
            "On Kaggle, add a GITHUB_TOKEN secret with read access to MIA-AI-Team/ApexID, "
            "or attach the repo as a dataset. "
            f"Expected branch: {GITHUB_BRANCH}."
        ) from exc
    if not _is_apexid_repo(destination):
        raise FileNotFoundError(f"Cloned {destination} but apexid/__init__.py is still missing.")
    return destination


def _list_input_mounts() -> list[str]:
    mounts = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.is_dir():
            continue
        try:
            names = sorted(path.name for path in root.iterdir())
        except OSError:
            names = []
        mounts.append(f"{root}: {names[:40]}")
    return mounts


def _find_manifest_file(root: Path, filename: str = "preparation_manifest.json", max_depth: int = 8) -> Path | None:
    skip_names = {
        ".git", "__pycache__", "images", "img", "photos", "embeddings",
        "checkpoints", "node_modules", "models", "pytorch",
    }
    skip_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".pt", ".bin", ".npz"}
    stack = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            entries = list(current.iterdir())
        except OSError:
            continue
        files = [entry for entry in entries if entry.is_file()]
        if any(entry.name == filename for entry in files):
            return current / filename
        if depth >= max_depth:
            continue
        bulky = sum(1 for entry in files if entry.suffix.lower() in skip_suffixes)
        if bulky >= 50:
            continue
        for entry in entries:
            if not entry.is_dir() or entry.name in skip_names or entry.name.startswith("."):
                continue
            stack.append((entry, depth + 1))
    return None


def _find_prepared_root() -> Path:
    explicit = [
        Path(os.environ.get("APEXID_PREPARED_ROOT", "")),
        Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
        Path("/kaggle/input/apexid-cvpr-prepared-bundle/apexid_cvpr_prepared"),
        Path("/kaggle/input/datasets/seifelkerdany/apexid-cvpr-prepared-bundle"),
        Path("/kaggle/input/apexid-cvpr-prepared-bundle"),
        Path("/kaggle/input/apexid-cvpr-prepared"),
        Path("/kaggle/input/apexid-cvpr-prepared/apexid_cvpr_prepared"),
        Path("/kaggle/input/apexid_cvpr_prepared"),
        Path("/kaggle/input/datasets/jamalhussien/apexid-cvpr-prepared"),
        Path("/kaggle/input/datasets/jamalhussien/apexid-cvpr-prepared/apexid_cvpr_prepared"),
        Path("/kaggle/input/datasets/mohamedbassat/apexid-cvpr-prepared"),
        Path("/kaggle/input/datasets/mohamedbassat/apexid-cvpr-prepared/apexid_cvpr_prepared"),
        Path("/kaggle/working/apexid_cvpr_prepared"),
        Path(r"D:\CVPR_BM\apexid_cvpr_prepared"),
        REPO_ROOT / "apexid_cvpr_prepared",
    ]
    for path in explicit:
        if str(path) and (path / "preparation_manifest.json").is_file():
            return path
    search_roots = []
    if ON_KAGGLE:
        search_roots.extend([Path("/kaggle/input"), Path("/kaggle/working")])
    search_roots.append(Path(r"D:\CVPR_BM"))
    for root in search_roots:
        if not root.is_dir():
            continue
        found = _find_manifest_file(root)
        if found is not None:
            return found.parent
    attached = "\n  ".join(_list_input_mounts()) or "(none)"
    raise FileNotFoundError(
        "Stage 00 prepared root with preparation_manifest.json was not found.\n"
        "Attach the local D:\\CVPR_BM\\apexid_cvpr_prepared folder as a Kaggle dataset "
        "(Add Input → Dataset), or set APEXID_PREPARED_ROOT to that dataset path.\n"
        f"Currently attached:\n  {attached}"
    )


REPO_ROOT = _ensure_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repo:", REPO_ROOT)

NOTEBOOK_PATH = REPO_ROOT / "notebooks" / "apexid-cvpr-01-extract-embeddings-calibrate-4models.ipynb"
PREPARED_ROOT = _find_prepared_root()
print("Prepared Stage 00:", PREPARED_ROOT)
OUTPUT_ROOT = (
    Path("/kaggle/working/apexid_cvpr_stage01_bhca_logistic")
    if ON_KAGGLE
    else Path(r"D:\CVPR_BM\apexid_cvpr_stage01_bhca_logistic")
)
EMBEDDINGS_ROOT = OUTPUT_ROOT / "embeddings"
EXISTING_STAGE01_ROOT = OUTPUT_ROOT

KAGGLE_MODEL_ROOT = Path("/kaggle/input/models/mohamedbassat")
MODEL_SPECS = {
    "ApexID_BHCA_A_v2": KAGGLE_MODEL_ROOT
    / "apexid-convnext-small-bhca-a-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_a_v2_rgb/best_backbone.pt",
    "ApexID_BHCA_SCA_v2": KAGGLE_MODEL_ROOT
    / "apexid-convnext-small-bhca-sca-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_sca_v2_rgb/best_backbone.pt",
    "ApexID_BHCA_SCSF2_v2": KAGGLE_MODEL_ROOT
    / "apexid-convnext-small-bhca-scsf2-v2-rgb/pytorch/default/1/apexid_convnext_small_bhca_scsf2_v2_rgb/best_backbone.pt",
}
MODEL_NAMES = tuple(MODEL_SPECS)
EXPECTED_MODELS = frozenset(MODEL_NAMES)
EXCLUDED_LEGACY_MODELS = frozenset({"ApexID_v1", "ApexID_HA_v2", "ApexID_v2", "MiewIDv3", "MegaDescriptor"})
REQUIRED_DATASETS = (
    "apexid_calibration",
    "bristol_gorillas",
    "elephants",
    "gcn_id",
    "jaguar_antonio",
    "spac_wild",
)

# Same FiftyOne Hugging Face source notebook 00 used. Frozen Stage 00 paths point
# at /kaggle/working/apexid_jaguar_materialized, which does not persist across sessions.
JAGUAR_HF_REPO = "JaguarCameraTrap/jaguar-camera-trap-curated-v1"
JAGUAR_HF_REVISION = os.environ.get("APEXID_JAGUAR_HF_REVISION")
JAGUAR_MATERIALIZED_ROOT = Path(
    os.environ.get(
        "APEXID_JAGUAR_MATERIALIZED_ROOT",
        "/kaggle/working/apexid_jaguar_materialized" if ON_KAGGLE else str(Path.cwd() / "apexid_jaguar_materialized"),
    )
)

EMBEDDINGS_ONLY = False
FORCE_REEXTRACT = False
EXTRACT_BATCH_SIZE = int(os.environ.get("APEXID_EXTRACT_BATCH_SIZE", "64"))
EXTRACT_NUM_WORKERS = int(os.environ.get("APEXID_EXTRACT_NUM_WORKERS", "2" if ON_KAGGLE else "0"))
EXTRACT_AMP = True
DATA_ROOT = Path(
    os.environ.get(
        "APEXID_DATA_ROOT",
        "/kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset" if ON_KAGGLE else ".",
    )
)

STAGE01_VERSION = "apexid-cvpr-stage01-bhca-extract-1.0.0"
CALIBRATION_PROTOCOL_VERSION = "cvpr_stage01_margin_gallery_logistic_v1"
SUPPORTED_REJECTION_CALIBRATION_METHODS = {"raw_max_cosine", "margin_gallery_logistic"}
REJECTION_CALIBRATION_METHOD = "margin_gallery_logistic"
LOGISTIC_FEATURE_NAMES = ("s1", "margin", "log_gallery_size")
EPISODES_PER_GALLERY_SIZE = 8
CALIBRATION_EPISODE_SEED = 42
LOGISTIC_C = 1.0
LOGISTIC_SOLVER = "lbfgs"
LOGISTIC_MAX_ITER = 2000
SEED = 42
N_JOBS = -1
SIMILARITY_CHUNK_SIZE = 512
CLUSTERING_THRESHOLD_CANDIDATES = None
L2_NORMALIZATION_ATOL = 5e-4
CREATE_STAGE02_COMPAT_LINKS = True

CPU_COUNT = os.cpu_count() or 1
NUMERIC_THREADS = CPU_COUNT if N_JOBS == -1 else max(1, min(int(N_JOBS), CPU_COUNT))
for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_name, str(NUMERIC_THREADS))

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score, normalized_mutual_info_score
try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None

CLUSTERING_THRESHOLD_CANDIDATES = np.linspace(-1.0, 1.0, 201)
random.seed(SEED)
np.random.seed(SEED)

try:
    import psutil
    AVAILABLE_RAM_GIB = psutil.virtual_memory().available / (1024 ** 3)
except Exception:
    AVAILABLE_RAM_GIB = None

if REJECTION_CALIBRATION_METHOD not in SUPPORTED_REJECTION_CALIBRATION_METHODS:
    raise ValueError(f"Unsupported rejection calibration method: {REJECTION_CALIBRATION_METHOD}")
if len(MODEL_NAMES) != 3:
    raise RuntimeError("This notebook is sealed to the three BHCA RGB backbones")

print("=" * 72)
print("ApexID CVPR Stage 01 — BHCA extraction + local calibration")
print("=" * 72)
print("Mode:\n    EMBEDDINGS_ONLY =", EMBEDDINGS_ONLY)
print("Repo:\n   ", REPO_ROOT)
print("Models:")
for _model, _weights in MODEL_SPECS.items():
    print(f"    {_model}")
    print(f"        {_weights}")
print("Prepared Stage 00:\n   ", PREPARED_ROOT)
print("Embedding caches:\n   ", EMBEDDINGS_ROOT)
print("New outputs:\n   ", OUTPUT_ROOT)
print("Embedding extraction:\n   ", "ENABLED" if not EMBEDDINGS_ONLY else "DISABLED")
print("Test split regeneration:\n    DISABLED")
print("Calibration source:\n    VALIDATION ONLY")
print("Final test labels used:\n    NO")
print("CPU count:\n   ", CPU_COUNT)
print("configured N_JOBS:\n   ", N_JOBS)
print("numeric worker threads:\n   ", NUMERIC_THREADS)
print("available RAM (GiB):\n   ", "unknown" if AVAILABLE_RAM_GIB is None else f"{AVAILABLE_RAM_GIB:.1f}")
print("=" * 72)


## 2. Safety contract and helpers

Missing or invalid caches raise an error that names the model, dataset, and expected path. After extraction, stored arrays are never silently changed. Cache normalization is measured during validation; the baseline cosine scorer then explicitly normalizes float32 blocks at score-computation time.

Sidecar YAML/JSON files are read from the same folder as `best_backbone.pt`. The output tree stores calibration results next to the extracted embedding caches. Lightweight hard links expose those caches for Stage 02 when the layout already matches.


In [ ]:
# ============================================================
# 2. Deterministic I/O, provenance, and cache inspection
# ============================================================

REQUIRED_NPZ_KEYS = frozenset({
    "embeddings", "image_ids", "paths", "identities", "dataset", "metadata_json"
})
runtime_rows = []


def stable_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, default=str)


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False, newline="") as handle:
        handle.write(text)
        temporary = Path(handle.name)
    os.replace(temporary, path)


def write_json(path: Path, value) -> None:
    atomic_write_text(path, json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False, default=str) + "\n")


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False, newline="") as handle:
        frame.to_csv(handle, index=False)
        temporary = Path(handle.name)
    os.replace(temporary, path)


def as_unicode_array(values) -> np.ndarray:
    """Pickle-free string array. pandas to_numpy() often yields dtype=object."""
    array = np.asarray(values)
    if array.dtype.kind in {"U", "S"}:
        return array
    if array.shape == ():
        return np.asarray(str(array.item()), dtype="U")
    flat = ["" if value is None else str(value) for value in array.ravel().tolist()]
    return np.asarray(flat, dtype="U").reshape(array.shape)


def same_str_array(left, right) -> bool:
    left_list = [str(value) for value in np.asarray(left).ravel().tolist()]
    right_list = [str(value) for value in np.asarray(right).ravel().tolist()]
    return left_list == right_list


def atomic_save_npz(path: Path, **arrays) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    cleaned = {}
    for key, value in arrays.items():
        array = np.asarray(value)
        if array.dtype.kind == "O":
            array = as_unicode_array(array)
        cleaned[key] = array
    with tempfile.NamedTemporaryFile("wb", dir=path.parent, delete=False, suffix=".npz") as handle:
        temporary = Path(handle.name)
    try:
        np.savez_compressed(temporary, **cleaned)
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def record_runtime(operation: str, started: float, **details) -> None:
    runtime_rows.append({
        "operation": operation,
        "elapsed_seconds": time.perf_counter() - started,
        **details,
    })


def cache_path(model_name: str, dataset_name: str) -> Path:
    return EMBEDDINGS_ROOT / model_name / dataset_name / "embeddings.npz"


def missing_cache_error(model_name: str, dataset_name: str, path: Path) -> FileNotFoundError:
    return FileNotFoundError(
        "Missing embedding cache:\n\n"
        f"Model: {model_name}\n"
        f"Dataset: {dataset_name}\n"
        f"Expected: {path}\n\n"
        "Re-run the BHCA extraction cell, or set FORCE_REEXTRACT=True."
    )


def _read_npy_header(archive: zipfile.ZipFile, member: str):
    with archive.open(member) as handle:
        version = np.lib.format.read_magic(handle)
        if version == (1, 0):
            shape, fortran, dtype = np.lib.format.read_array_header_1_0(handle)
        else:
            shape, fortran, dtype = np.lib.format.read_array_header_2_0(handle)
    return tuple(shape), bool(fortran), np.dtype(dtype)


def npz_has_object_arrays(path: Path) -> bool:
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith(".npy"):
                continue
            _shape, _fortran, dtype = _read_npy_header(archive, name)
            if getattr(dtype, "hasobject", False) or dtype.kind == "O":
                return True
    return False


def rewrite_npz_object_strings(path: Path) -> bool:
    """Convert object-string NPZ keys to unicode so allow_pickle=False can load them.

    Does not re-extract embeddings. Safe for caches this notebook just wrote.
    """
    path = Path(path)
    if not path.is_file() or not npz_has_object_arrays(path):
        return False
    with np.load(path, allow_pickle=True) as cached:
        payload = {key: np.array(cached[key]) for key in cached.files}
    atomic_save_npz(path, **payload)
    print(f"Rewrote object-string NPZ to unicode (no re-extract): {path}", flush=True)
    return True


def inspect_npz_header(path: Path) -> dict:
    rewrite_npz_object_strings(path)
    try:
        with zipfile.ZipFile(path) as archive:
            members = {name[:-4]: name for name in archive.namelist() if name.endswith(".npy")}
            missing = REQUIRED_NPZ_KEYS - set(members)
            if missing:
                raise ValueError(f"missing .npz keys: {sorted(missing)}")
            shape, fortran, dtype = _read_npy_header(archive, members["embeddings"])
        with np.load(path, allow_pickle=False) as cached:
            metadata = json.loads(str(cached["metadata_json"].item()))
            dataset_values = cached["dataset"].astype(str)
        return {
            "shape": shape,
            "fortran_order": fortran,
            "dtype": dtype,
            "metadata": metadata,
            "dataset_values": dataset_values,
            "keys": tuple(sorted(members)),
        }
    except Exception as exc:
        raise RuntimeError(f"Embedding cache cannot be inspected: {path}\n{type(exc).__name__}: {exc}") from exc


def normalize_for_cosine(values: np.ndarray, *, context: str) -> np.ndarray:
    """Explicit float32 normalization used only by cosine score computation."""
    result = np.asarray(values, dtype=np.float32)
    if result.ndim != 2:
        raise ValueError(f"{context}: expected a 2D array")
    norms = np.linalg.norm(result, axis=1, keepdims=True)
    if np.any(norms <= 1e-12) or not np.isfinite(norms).all():
        raise ValueError(f"{context}: zero or non-finite norm")
    return result / norms


def link_cache_for_stage02(source: Path, destination: Path) -> str:
    """Make a no-copy Stage 02 view; never copy a multi-GB embedding cache."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        try:
            if os.path.samefile(source, destination):
                return "existing_hardlink_or_samefile"
        except OSError:
            pass
        raise FileExistsError(f"Refusing to replace unrelated Stage 02 cache view: {destination}")
    try:
        os.link(source, destination)
        return "hardlink"
    except OSError as hardlink_error:
        try:
            destination.symlink_to(source)
            return "symlink"
        except OSError as symlink_error:
            return f"unavailable: hardlink={hardlink_error}; symlink={symlink_error}"


INVALID_IDENTITIES = {"", "unknown", "none", "nan", "null", "new_whale"}


def _decode_fiftyone_mask(mask_field):
    import base64
    import io
    import zlib

    if mask_field is None:
        raise ValueError("Empty FiftyOne mask")
    if isinstance(mask_field, dict):
        binary = mask_field.get("$binary", mask_field)
        encoded = binary.get("base64") if isinstance(binary, dict) else binary
    else:
        encoded = mask_field
    raw = base64.b64decode(encoded)
    try:
        raw = zlib.decompress(raw)
    except zlib.error:
        pass
    return np.asarray(np.load(io.BytesIO(raw), allow_pickle=False), dtype=bool)


def _composite_black_background(image, sample):
    from PIL import Image

    detections = (sample.get("segmentations_body") or {}).get("detections") or []
    if not detections:
        raise ValueError("No segmentations_body detections")
    rgb = image.convert("RGB")
    width, height = rgb.size
    pixels = np.asarray(rgb)
    full_mask = np.zeros((height, width), dtype=bool)
    for detection in detections:
        if detection.get("mask") is None or detection.get("bounding_box") is None:
            continue
        mask = np.squeeze(_decode_fiftyone_mask(detection["mask"]))
        if mask.ndim != 2:
            raise ValueError("FiftyOne mask is not two-dimensional")
        bx, by, _bw, _bh = map(float, detection["bounding_box"])
        x1, y1 = int(round(bx * width)), int(round(by * height))
        x2, y2 = min(width, x1 + mask.shape[1]), min(height, y1 + mask.shape[0])
        if x2 > x1 and y2 > y1:
            full_mask[y1:y2, x1:x2] |= mask[: y2 - y1, : x2 - x1]
    if not full_mask.any():
        raise ValueError("Composed segmentation mask is empty")
    output = np.zeros_like(pixels)
    output[full_mask] = pixels[full_mask]
    return Image.fromarray(output)


def resolve_hf_dataset_revision(
    *, repo_id: str, requested_revision: str | None, lock_path: Path, token: str | None,
) -> str:
    """Resolve a dataset once, then reuse its recorded immutable commit on reruns."""
    lock_path = Path(lock_path)
    requested_revision = requested_revision or None
    if lock_path.is_file():
        record = json.loads(lock_path.read_text(encoding="utf-8"))
        if record.get("repo_id") == repo_id and (
            requested_revision is None
            or requested_revision in {record.get("requested_revision"), record.get("revision")}
        ):
            revision = str(record.get("revision", ""))
            if len(revision) == 40 and all(value in "0123456789abcdef" for value in revision):
                return revision
    from huggingface_hub import HfApi

    try:
        info = HfApi(token=token).dataset_info(repo_id, revision=requested_revision or "main")
    except Exception as exc:
        raise RuntimeError(
            f"Cannot access Hugging Face dataset {repo_id}. Enable the Kaggle hugging_face "
            "secret with dataset read access."
        ) from exc
    revision = str(info.sha)
    if len(revision) != 40 or any(value not in "0123456789abcdef" for value in revision):
        raise RuntimeError(f"Dataset {repo_id} did not resolve to an immutable commit")
    write_json(lock_path, {
        "repo_id": repo_id,
        "requested_revision": requested_revision,
        "revision": revision,
        "selection": "resolved before calibration and external evaluation",
    })
    return revision


def materialize_jaguar_antonio(
    *,
    hf_repo: str,
    hf_revision: str,
    output_root: Path,
    token: str | None,
) -> Path:
    """Materialize deterministic mask-based Jaguar Antonio images from FiftyOne JSON."""
    from PIL import Image

    if not hf_revision or len(hf_revision) < 12:
        raise ValueError("Jaguar dataset must be pinned to an immutable Hugging Face revision")
    try:
        from huggingface_hub import snapshot_download
    except ImportError as exc:
        raise ImportError("Install huggingface-hub to materialize Jaguar Antonio") from exc
    if not token:
        raise RuntimeError(
            "Jaguar Antonio is unavailable locally and no Hugging Face token was supplied. "
            "Enable the Kaggle `hugging_face` secret / HF_TOKEN."
        )
    output_root = Path(output_root)
    manifest = output_root / "manifest.csv"
    provenance_path = output_root / "materialization_manifest.json"
    if provenance_path.is_file():
        previous = json.loads(provenance_path.read_text(encoding="utf-8"))
        if previous.get("hf_repo") != hf_repo or previous.get("hf_revision") != hf_revision:
            raise RuntimeError("Jaguar materialization revision changed; choose a new output directory")
        if manifest.is_file() and previous.get("manifest_sha256") == sha256_file(manifest):
            print(f"CACHE HIT — verified Jaguar materialization: {manifest}", flush=True)
            return manifest
    snapshot = Path(
        snapshot_download(
            repo_id=hf_repo,
            repo_type="dataset",
            revision=hf_revision,
            token=token,
            local_dir=output_root / "hf_snapshot",
        )
    )
    samples_files = sorted(snapshot.rglob("samples.json"))
    if not samples_files:
        raise FileNotFoundError(f"No samples.json found in pinned Jaguar snapshot {snapshot}")
    payload = json.loads(samples_files[0].read_text(encoding="utf-8"))
    samples = payload.get("samples", payload) if isinstance(payload, dict) else payload
    rows = []
    failures = Counter()
    for index, sample in enumerate(samples):
        identity = str(sample.get("jaguar_id") or "").strip()
        if identity.lower() in INVALID_IDENTITIES:
            failures["missing_identity"] += 1
            continue
        relative = str(sample.get("filepath") or sample.get("path") or "")
        candidates = [Path(relative), snapshot / relative]
        source = next((path for path in candidates if path.is_file()), None)
        if source is None:
            matches = sorted(snapshot.rglob(Path(relative).name)) if relative else []
            source = matches[0] if matches else None
        if source is None:
            failures["missing_image"] += 1
            continue
        output = output_root / "images" / identity.replace("/", "_") / f"{index:08d}.jpg"
        output.parent.mkdir(parents=True, exist_ok=True)
        try:
            with Image.open(source) as image:
                composite = _composite_black_background(image, sample)
                if not output.is_file():
                    composite.save(output, quality=95, subsampling=0)
        except Exception:
            failures["mask_failure"] += 1
            continue
        rows.append(
            {
                "identity": identity,
                "path": str(output.resolve()),
                "source_filepath": relative,
                "segmented_black_bg": True,
            }
        )
    if not rows:
        raise RuntimeError(f"Jaguar materialization produced no images; failures={dict(failures)}")
    atomic_write_csv(manifest, pd.DataFrame(rows))
    write_json(
        provenance_path,
        {
            "hf_repo": hf_repo,
            "hf_revision": hf_revision,
            "samples_json_sha256": sha256_file(samples_files[0]),
            "manifest_sha256": sha256_file(manifest),
            "rows": len(rows),
            "failures": dict(failures),
        },
    )
    print(
        f"Jaguar Antonio materialized {len(rows)} FiftyOne black-background images at {output_root} "
        f"(failures={dict(failures)})",
        flush=True,
    )
    return manifest


def _looks_like_git_sha(value) -> bool:
    text = str(value or "")
    return len(text) == 40 and all(char in "0123456789abcdef" for char in text)


def _revision_from_mapping(payload) -> str | None:
    if not isinstance(payload, dict):
        return None
    for key in ("revision", "hf_revision"):
        value = payload.get(key)
        if _looks_like_git_sha(value):
            return str(value)
    source = payload.get("source")
    if isinstance(source, dict):
        nested = _revision_from_mapping(source)
        if nested:
            return nested
    return None


def jaguar_revision_from_stage00() -> str | None:
    records = [
        PREPARED_ROOT / "manifests" / "jaguar_revision_lock.json",
        PREPARED_ROOT / "jaguar_revision_lock.json",
        PREPARED_ROOT / "manifests" / "jaguar_antonio_preparation_record.json",
        PREPARED_ROOT / "tables" / "jaguar_antonio_preparation_record.json",
    ]
    for path in records:
        if not path.is_file():
            continue
        revision = _revision_from_mapping(json.loads(path.read_text(encoding="utf-8")))
        if revision:
            return revision
    sources = ((PREPARATION_MANIFEST.get("inputs") or {}).get("external_sources") or {})
    jaguar_source = sources.get("jaguar_antonio") if isinstance(sources, dict) else None
    if jaguar_source is None and isinstance(sources, list):
        jaguar_source = next(
            (
                item for item in sources
                if isinstance(item, dict) and item.get("dataset") == "jaguar_antonio"
            ),
            None,
        )
    revision = _revision_from_mapping(jaguar_source)
    if revision:
        return revision
    env_revision = os.environ.get("APEXID_JAGUAR_HF_REVISION") or JAGUAR_HF_REVISION
    return str(env_revision) if env_revision else None


def ensure_jaguar_antonio_materialized() -> Path:
    """Rebuild the Stage 00 Jaguar working copy from the FiftyOne HF dataset if needed."""
    frame = canonical_frames["jaguar_antonio"]
    sample = Path(str(frame["path"].iloc[0]))
    if sample.is_file():
        print("Jaguar Antonio frozen path already present:", sample)
        return sample.parent.parent
    output_root = JAGUAR_MATERIALIZED_ROOT
    output_root.mkdir(parents=True, exist_ok=True)
    try:
        import huggingface_hub  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    token = _hf_token()
    requested = jaguar_revision_from_stage00() or JAGUAR_HF_REVISION
    revision = resolve_hf_dataset_revision(
        repo_id=JAGUAR_HF_REPO,
        requested_revision=requested,
        lock_path=OUTPUT_ROOT / "jaguar_revision_lock.json",
        token=token,
    )
    print("Jaguar Antonio FiftyOne HF repo:", JAGUAR_HF_REPO)
    print("Jaguar Antonio pinned revision:", revision)
    materialize_jaguar_antonio(
        hf_repo=JAGUAR_HF_REPO,
        hf_revision=revision,
        output_root=output_root,
        token=token,
    )
    if not sample.is_file():
        raise FileNotFoundError(
            "Jaguar rematerialization finished but the frozen Stage 00 path is still missing: "
            f"{sample}. The Hugging Face revision must match the Stage 00 freeze."
        )
    return output_root


print("Safety helpers ready.")


## 3. Startup preflight and frozen manifest loading

This preflight validates the configured paths, the three BHCA checkpoints, required datasets, and Stage 00 calibration roles before extraction or clustering. Legacy HA / ApexID_v2 / Miew / Mega cache directories are ignored if they happen to sit nearby.


In [ ]:
# ============================================================
# 3. Strong startup preflight (no image access, no inference)
# ============================================================

if not NOTEBOOK_PATH.is_file():
    print(f"NOTEBOOK_PATH is not inside the cloned repo (fine on Kaggle): {NOTEBOOK_PATH}")
if not PREPARED_ROOT.is_dir():
    raise FileNotFoundError(f"Frozen Stage 00 root does not exist: {PREPARED_ROOT}")
EMBEDDINGS_ROOT.mkdir(parents=True, exist_ok=True)
if OUTPUT_ROOT == NOTEBOOK_PATH or OUTPUT_ROOT.suffix.lower() == ".ipynb":
    raise ValueError("OUTPUT_ROOT must be an artifact directory, not the notebook file")

for directory in (
    OUTPUT_ROOT,
    OUTPUT_ROOT / "calibration",
    OUTPUT_ROOT / "scores",
    OUTPUT_ROOT / "tables",
    OUTPUT_ROOT / "manifests",
    OUTPUT_ROOT / "logs",
):
    directory.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=OUTPUT_ROOT, prefix=".write_test_", delete=True):
    pass

preparation_manifest_path = PREPARED_ROOT / "preparation_manifest.json"
if not preparation_manifest_path.is_file():
    raise FileNotFoundError(f"Stage 00 preparation manifest missing: {preparation_manifest_path}")
PREPARATION_MANIFEST = json.loads(preparation_manifest_path.read_text(encoding="utf-8"))
if PREPARATION_MANIFEST.get("status") != "passed":
    raise RuntimeError("Stage 00 preparation manifest is not marked passed")
split_v2 = dict(PREPARATION_MANIFEST.get("apexid_split_v2", {}))
if not split_v2.get("passed"):
    raise RuntimeError("Frozen ApexID Split V2 integrity record is not marked passed")

DATASET_MANIFEST_HASHES = dict(PREPARATION_MANIFEST["dataset_manifest_hashes"])
if set(DATASET_MANIFEST_HASHES) != set(REQUIRED_DATASETS):
    raise RuntimeError(
        f"Stage 00 dataset set mismatch: {sorted(DATASET_MANIFEST_HASHES)} != {sorted(REQUIRED_DATASETS)}"
    )

manifest_paths = {
    "apexid_calibration": PREPARED_ROOT / "manifests" / "apexid_calibration_manifest.csv",
    **{
        name: PREPARED_ROOT / "manifests" / f"{name}_canonical_manifest.csv"
        for name in REQUIRED_DATASETS if name != "apexid_calibration"
    },
}
canonical_frames = {}
for dataset_name, manifest_path in manifest_paths.items():
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Frozen canonical manifest missing: {manifest_path}")
    frame = pd.read_csv(manifest_path, dtype=str).fillna("")
    required = {"image_id", "path", "identity", "dataset"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{dataset_name}: manifest lacks {sorted(required - set(frame.columns))}")
    if frame["image_id"].duplicated().any():
        raise ValueError(f"{dataset_name}: duplicate image_id values in frozen manifest")
    if set(frame["dataset"].astype(str)) != {dataset_name}:
        raise ValueError(f"{dataset_name}: dataset column does not match the manifest identity")
    canonical_frames[dataset_name] = frame.reset_index(drop=True)

calibration_frame = canonical_frames["apexid_calibration"]
if "calibration_role" not in calibration_frame.columns:
    raise ValueError("Calibration manifest lacks calibration_role")
expected_roles = {"train_gallery", "val_known", "val_unknown"}
actual_roles = set(calibration_frame["calibration_role"].astype(str))
if actual_roles != expected_roles:
    raise ValueError(f"Calibration roles changed: {sorted(actual_roles)} != {sorted(expected_roles)}")
known_ids = set(calibration_frame.loc[calibration_frame["calibration_role"].eq("val_known"), "identity"])
unknown_ids = set(calibration_frame.loc[calibration_frame["calibration_role"].eq("val_unknown"), "identity"])
gallery_ids = set(calibration_frame.loc[calibration_frame["calibration_role"].eq("train_gallery"), "identity"])
if not known_ids.issubset(gallery_ids):
    raise RuntimeError("Validation-known identities are not a subset of the frozen gallery identities")
if unknown_ids & gallery_ids:
    raise RuntimeError("Validation-unknown identities overlap the frozen gallery identities")

print("Active model set:", list(MODEL_NAMES))
print("Final-test manifests loaded for calibration: NO")
print("STARTUP PREFLIGHT: PASSED")


## 3b. Extract BHCA embeddings

Each backbone is loaded with `ApexIDModelAdapter`. Architecture and eval transforms come from `config_resolved.yaml` (or `config.yaml`) beside `best_backbone.pt`; `best_metadata.json` is recorded as provenance only. Heads are discarded. Images follow the frozen Stage 00 manifests. Jaguar Antonio is the FiftyOne Hugging Face dataset (`JaguarCameraTrap/jaguar-camera-trap-curated-v1`); this cell rematerializes the same black-background composites notebook 00 wrote to `/kaggle/working/apexid_jaguar_materialized`. Completed `.npz` caches are reused unless `FORCE_REEXTRACT=True`.


In [ ]:
# ============================================================
# 3b. Extract L2 embeddings from the three BHCA backbones
# ============================================================

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from apexid.data import build_eval_transforms
from apexid.evaluation.benchmark_evaluator import ApexIDModelAdapter, BenchmarkImageDataset
from apexid.utils.config import load_config

SIDECAR_CONFIG_NAMES = (
    "config_resolved.yaml",
    "config_resolved.yml",
    "config.yaml",
    "config.yml",
)
SIDECAR_METADATA_NAMES = (
    "best_metadata.json",
    "metadata.json",
)

# Kaggle model datasets often upload only best_backbone.pt (+ JSON metadata).
# RGB BHCA-v2 used ImageNet RGB normalization, not ApexID dataset-stat gray/RGB stats.
RGB_BHCA_EVAL_CONFIG = {
    "seed": 42,
    "data": {
        "image_size": [440, 440],
        "wavelet": False,
        "grayscale": False,
        "transforms": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
        },
    },
    "model": {
        "name": "apexid",
        "embedding_dim": 2048,
        "input_channels": 3,
        "backbone": {
            "name": "convnext_small_pt",
            "kwargs": {"pretrained": True},
        },
    },
}


# Frozen Stage 00 manifests keep original absolute paths. These are the live
# Kaggle image parents for this session. Split V2 CSVs are already consumed
# inside the Seif prepared bundle; they are not image roots.
# Jaguar is NOT wildlife-reid: notebook 00 materializes it from FiftyOne HF.
APEXID_SPLIT_V2_ROOT = Path("/kaggle/input/datasets/seifelkerdany/apexid-split-v2")
KNOWN_IMAGE_PARENTS = (
    Path("/kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset/images"),
    Path("/kaggle/input/datasets/jamalhussien/bristol-gorillas-sam3-black-background/images"),
    Path("/kaggle/input/datasets/jamalhussien/apexid-wildlife-reid-test-data/images"),
    JAGUAR_MATERIALIZED_ROOT / "images",
)
KNOWN_DATASET_ROOTS = (
    Path("/kaggle/input/datasets/seifelkerdany/apexid-dataset/apexid-dataset"),
    Path("/kaggle/input/datasets/jamalhussien/bristol-gorillas-sam3-black-background"),
    Path("/kaggle/input/datasets/jamalhussien/apexid-wildlife-reid-test-data"),
    JAGUAR_MATERIALIZED_ROOT,
)
JAGUAR_IMAGE_PARENTS = (
    JAGUAR_MATERIALIZED_ROOT / "images",
)
IMAGE_ROOT_REWRITES: list[tuple[str, str]] = []


def _posix_path(path) -> str:
    return str(path).replace("\\", "/").rstrip("/")


def _remember_prefix_rewrite(frozen_path: str, found: Path) -> None:
    frozen_parts = Path(_posix_path(frozen_path)).parts
    found_parts = Path(_posix_path(found)).parts
    shared = 0
    while (
        shared < min(len(frozen_parts), len(found_parts))
        and frozen_parts[-(shared + 1)] == found_parts[-(shared + 1)]
    ):
        shared += 1
    if shared < 1:
        return
    old_prefix = _posix_path(Path(*frozen_parts[:-shared])) if len(frozen_parts) > shared else ""
    new_prefix = _posix_path(Path(*found_parts[:-shared])) if len(found_parts) > shared else ""
    if not old_prefix or not new_prefix:
        return
    pair = (old_prefix, new_prefix)
    if pair not in IMAGE_ROOT_REWRITES:
        IMAGE_ROOT_REWRITES.append(pair)
        print(f"Image root remap:\n    {old_prefix}\n -> {new_prefix}")


def _candidate_image_paths(frozen_path: str) -> list[Path]:
    posix = _posix_path(frozen_path)
    frozen = Path(posix)
    candidates: list[Path] = [frozen]
    for old, new in IMAGE_ROOT_REWRITES:
        if posix == old or posix.startswith(old + "/"):
            candidates.append(Path(new + posix[len(old):]))
    tail = posix.split("/images/", 1)[1] if "/images/" in posix else "/".join(frozen.parts[-2:])
    identity = frozen.parent.name
    filename = frozen.name
    is_jaguar = "apexid_jaguar_materialized" in posix or "/jaguar" in posix.lower()
    parents = JAGUAR_IMAGE_PARENTS if is_jaguar else KNOWN_IMAGE_PARENTS
    roots = (JAGUAR_MATERIALIZED_ROOT,) if is_jaguar else KNOWN_DATASET_ROOTS
    for parent in (*parents, *(root / "images" for root in roots)):
        candidates.extend(
            [
                parent / tail,
                parent / identity / filename,
                parent / filename,
            ]
        )
    for root in roots:
        candidates.extend(
            [
                root / tail,
                root / "images" / identity / filename,
                root / identity / filename,
            ]
        )
    unique = []
    seen = set()
    for candidate in candidates:
        key = _posix_path(candidate)
        if key in seen:
            continue
        seen.add(key)
        unique.append(Path(key))
    return unique


def resolve_source_image_path(frozen_path: str) -> Path:
    for candidate in _candidate_image_paths(frozen_path):
        if candidate.is_file():
            if _posix_path(candidate) != _posix_path(frozen_path):
                _remember_prefix_rewrite(frozen_path, candidate)
            return candidate
    raise FileNotFoundError(f"Could not read benchmark image: {frozen_path}")


def install_image_root_rewrites() -> None:
    global IMAGE_ROOT_REWRITES
    IMAGE_ROOT_REWRITES = []
    print("Split V2 CSVs (already inside Stage 00 bundle, not opened here):", APEXID_SPLIT_V2_ROOT)
    print("Live image parents:")
    for parent in KNOWN_IMAGE_PARENTS:
        print(f"    {'OK' if parent.is_dir() else 'missing':<8} {parent}")
    missing = []
    for dataset_name, frame in canonical_frames.items():
        sample = str(frame["path"].iloc[0])
        try:
            found = resolve_source_image_path(sample)
        except FileNotFoundError:
            missing.append((dataset_name, sample))
            print(f"    FAIL {dataset_name:<20} {sample}")
            continue
        status = "frozen" if _posix_path(found) == _posix_path(sample) else "remapped"
        print(f"    {status:<8} {dataset_name:<20} {found}")
    if missing:
        details = "\n".join(f"    {name}: {path}" for name, path in missing)
        raise FileNotFoundError(
            "Could not locate source images for one or more frozen manifests.\n"
            "Attach ApexID, Bristol SAM3, and wildlife Re-ID images, and enable the "
            "Kaggle hugging_face secret so Jaguar can rematerialize from FiftyOne HF.\n"
            + details
        )


class PathRewritingImageDataset(BenchmarkImageDataset):
    def __getitem__(self, index: int):
        import cv2

        row = self.frame.iloc[index]
        frozen_path = str(row.path)
        candidate = resolve_source_image_path(frozen_path)
        image = cv2.imread(str(candidate), cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(
                f"Could not read benchmark image: {candidate} (frozen path {frozen_path})"
            )
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        transformed = self.transform(image=image)["image"]
        return transformed, str(row.image_id), frozen_path


def _search_sidecar(folders, names=None, suffixes=None):
    seen = set()
    for folder in folders:
        if folder is None or not folder.is_dir():
            continue
        resolved = folder.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if names:
            for name in names:
                candidate = resolved / name
                if candidate.is_file():
                    return candidate
        if suffixes:
            matches = sorted(
                path for path in resolved.iterdir()
                if path.is_file() and path.suffix.lower() in suffixes
            )
            if matches:
                return matches[0]
    return None


def resolve_model_bundle(model_name: str) -> dict:
    weights = Path(MODEL_SPECS[model_name])
    if not weights.is_file():
        raise FileNotFoundError(f"{model_name}: missing backbone weights: {weights}")
    folder = weights.parent
    search_folders = [folder, folder.parent, folder.parent.parent]
    config_path = _search_sidecar(search_folders, names=SIDECAR_CONFIG_NAMES, suffixes={".yaml", ".yml"})
    metadata_path = _search_sidecar(search_folders, names=SIDECAR_METADATA_NAMES, suffixes={".json"})
    sidecar_metadata = {}
    if metadata_path is not None:
        try:
            payload = json.loads(metadata_path.read_text(encoding="utf-8"))
            if isinstance(payload, dict):
                sidecar_metadata = payload
        except Exception:
            sidecar_metadata = {"metadata_path": str(metadata_path)}
    if config_path is not None:
        model_cfg = load_config(config_path=config_path)
        config_source = str(config_path)
    else:
        try:
            present = sorted(path.name for path in folder.iterdir())
        except OSError:
            present = []
        print(
            f"{model_name}: no YAML beside {folder}. "
            f"Files: {present}. Using RGB ConvNeXt-Small eval fallback (ImageNet mean/std)."
        )
        model_cfg = load_config(config=RGB_BHCA_EVAL_CONFIG)
        config_source = "rgb_bhca_eval_fallback_imagenet"
    return {
        "model": model_name,
        "weights": weights.resolve(),
        "folder": folder.resolve(),
        "config": model_cfg,
        "config_path": None if config_path is None else config_path.resolve(),
        "config_source": config_source,
        "metadata_path": None if metadata_path is None else metadata_path.resolve(),
        "sidecar_metadata": sidecar_metadata,
    }


def preprocessing_signature(model_cfg: dict, bundle: dict) -> dict:
    data_cfg = dict(model_cfg.get("data") or {})
    transforms_cfg = dict(data_cfg.get("transforms") or {})
    model_section = dict(model_cfg.get("model") or {})
    config_path = bundle.get("config_path")
    config_hash = sha256_file(config_path) if config_path else sha256_bytes(stable_json(model_cfg).encode("utf-8"))
    return {
        "config_sha256": config_hash,
        "config_source": bundle.get("config_source", "unknown"),
        "image_size": data_cfg.get("image_size"),
        "implementation": "apexid.data.build_eval_transforms",
        "input_channels": int(model_section.get("input_channels") or (6 if data_cfg.get("wavelet") else 3)),
        "mean": transforms_cfg.get("mean"),
        "std": transforms_cfg.get("std"),
        "wavelet": bool(data_cfg.get("wavelet", False)),
        "grayscale": bool(data_cfg.get("grayscale", False)),
    }


def cache_metadata(model_name: str, dataset_name: str, frame: pd.DataFrame, bundle: dict, adapter, amp_enabled: bool) -> dict:
    image_ids = frame["image_id"].astype(str).tolist()
    return {
        "model": model_name,
        "dataset": dataset_name,
        "model_identifier": str(bundle["weights"]),
        "model_revision": bundle["sidecar_metadata"].get("best_epoch", "unknown"),
        "checkpoint_sha256": adapter.checkpoint_metadata["weights_sha256"],
        "config_path": bundle.get("config_source") or str(bundle.get("config_path") or ""),
        "metadata_path": None if bundle["metadata_path"] is None else str(bundle["metadata_path"]),
        "number_images": int(len(frame)),
        "embedding_dimension": int(adapter.checkpoint_metadata["embedding_dim"]),
        "l2_normalized": True,
        "amp_enabled": bool(amp_enabled),
        "dataset_manifest_sha256": DATASET_MANIFEST_HASHES[dataset_name],
        "image_order_sha256": sha256_bytes("\n".join(image_ids).encode("utf-8")),
        "preprocessing_signature": preprocessing_signature(adapter.config, bundle),
        "sidecar_metadata": bundle["sidecar_metadata"],
    }


def write_embedding_cache(model_name: str, dataset_name: str, embeddings: np.ndarray, frame: pd.DataFrame, metadata: dict) -> Path:
    path = cache_path(model_name, dataset_name)
    atomic_save_npz(
        path,
        embeddings=np.asarray(embeddings, dtype=np.float32),
        image_ids=as_unicode_array(frame["image_id"].astype(str).tolist()),
        paths=as_unicode_array(frame["path"].astype(str).tolist()),
        identities=as_unicode_array(frame["identity"].astype(str).tolist()),
        dataset=np.asarray([dataset_name], dtype="U"),
        metadata_json=np.asarray(stable_json(metadata), dtype="U"),
    )
    return path


def extract_dataset_embeddings(model_name: str, dataset_name: str, adapter, transform, data_root: Path) -> np.ndarray:
    frame = canonical_frames[dataset_name]
    dataset = PathRewritingImageDataset(frame, data_root, transform)
    loader = DataLoader(
        dataset,
        batch_size=EXTRACT_BATCH_SIZE,
        shuffle=False,
        num_workers=EXTRACT_NUM_WORKERS,
        pin_memory=adapter.device.type == "cuda",
        persistent_workers=EXTRACT_NUM_WORKERS > 0,
    )
    chunks = []
    image_ids = []
    paths = []
    started = time.perf_counter()
    with torch.inference_mode():
        for images, batch_ids, batch_paths in tqdm(loader, desc=f"{model_name}/{dataset_name}", leave=False):
            images = images.to(adapter.device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=adapter.amp):
                output = adapter.model(images)
            output = torch.nn.functional.normalize(output.float(), dim=1)
            if not torch.isfinite(output).all():
                raise ValueError(f"{model_name}/{dataset_name}: model produced NaN or Inf embeddings")
            chunks.append(output.cpu().numpy().astype(np.float32, copy=False))
            image_ids.extend(map(str, batch_ids))
            paths.extend(map(str, batch_paths))
    embeddings = np.concatenate(chunks, axis=0) if chunks else np.empty((0, adapter.model.embedding_dim), dtype=np.float32)
    expected_ids = frame["image_id"].astype(str).tolist()
    expected_paths = frame["path"].astype(str).tolist()
    if image_ids != expected_ids or paths != expected_paths:
        raise RuntimeError(f"{model_name}/{dataset_name}: extraction order no longer matches the frozen manifest")
    record_runtime("embedding_extraction", started, model=model_name, dataset=dataset_name, num_images=len(frame))
    return embeddings


MODEL_BUNDLES = {name: resolve_model_bundle(name) for name in MODEL_NAMES}
print("Resolved BHCA bundles")
for name, bundle in MODEL_BUNDLES.items():
    print(f"  {name}")
    print(f"    weights : {bundle['weights']}")
    print(f"    config  : {bundle['config_source']}")
    print(f"    metadata: {bundle['metadata_path']}")

ensure_jaguar_antonio_materialized()
install_image_root_rewrites()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Extraction device:", device)
if device.type != "cuda":
    print("WARNING: CUDA is unavailable; BHCA extraction will run on CPU.")

for model_index, model_name in enumerate(MODEL_NAMES, start=1):
    missing_or_forced = []
    for dataset_name in REQUIRED_DATASETS:
        path = cache_path(model_name, dataset_name)
        if FORCE_REEXTRACT or not path.is_file():
            missing_or_forced.append(dataset_name)
    if not missing_or_forced:
        print(f"[{model_index}/{len(MODEL_NAMES)}] {model_name}: all six caches present, skipping extraction")
        continue

    bundle = MODEL_BUNDLES[model_name]
    model_cfg = bundle["config"]
    adapter = ApexIDModelAdapter(model_cfg, device, amp=EXTRACT_AMP and device.type == "cuda")
    adapter.load_weights(bundle["weights"])
    transform = build_eval_transforms(config=model_cfg)
    print(
        f"[{model_index}/{len(MODEL_NAMES)}] {model_name}: "
        f"dim={adapter.checkpoint_metadata['embedding_dim']} "
        f"extracting {len(missing_or_forced)} dataset(s)"
    )
    for dataset_name in missing_or_forced:
        embeddings = extract_dataset_embeddings(model_name, dataset_name, adapter, transform, DATA_ROOT)
        metadata = cache_metadata(model_name, dataset_name, canonical_frames[dataset_name], bundle, adapter, adapter.amp)
        write_embedding_cache(model_name, dataset_name, embeddings, canonical_frames[dataset_name], metadata)
        print(f"    wrote {dataset_name:<20} {embeddings.shape[0]:>7,} x {embeddings.shape[1]}")
        del embeddings
        gc.collect()
    del adapter
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("BHCA EMBEDDING EXTRACTION: COMPLETE")


## 4. Rigorous cache validation and embedding inventory

Every required cache is opened once before calibration. The checks cover numeric/finite 2D embeddings, row counts, dimensions, unique row identifiers, exact identifier/label/path alignment with the frozen Stage 00 canonical manifests, dataset/model provenance, manifest hashes, and L2 norms. Cache files are hashed for reproducibility.

This pass does not open source images. It does not rewrite or renormalize stored embeddings.

Existing caches that stored pandas strings as `dtype=object` are rewritten in place to unicode. Do not set `FORCE_REEXTRACT=True` for that error.


In [ ]:
# ============================================================
# 4. Full cache validation — sequential and memory bounded
# ============================================================

if "rewrite_npz_object_strings" not in globals() or "as_unicode_array" not in globals():
    raise RuntimeError(
        "Re-run the helpers cell (section 2) first. Existing embedding caches will be "
        "rewritten in place from object-strings to unicode. Do not re-extract and do "
        "not set FORCE_REEXTRACT=True."
    )

def validate_embedding_cache(model_name: str, dataset_name: str, expected_frame: pd.DataFrame) -> dict:
    path = cache_path(model_name, dataset_name)
    if not path.is_file():
        raise missing_cache_error(model_name, dataset_name, path)
    started = time.perf_counter()
    try:
        rewrite_npz_object_strings(path)
        with np.load(path, allow_pickle=False) as cached:
            missing = REQUIRED_NPZ_KEYS - set(cached.files)
            if missing:
                raise ValueError(f"missing keys {sorted(missing)}")
            embeddings = cached["embeddings"]
            image_ids = cached["image_ids"].astype(str)
            stored_paths = cached["paths"].astype(str)
            identities = cached["identities"].astype(str)
            dataset_values = cached["dataset"].astype(str)
            metadata = json.loads(str(cached["metadata_json"].item()))

        if embeddings.ndim != 2 or embeddings.shape[0] <= 0 or embeddings.shape[1] <= 0:
            raise ValueError(f"invalid embedding shape {embeddings.shape}")
        if not np.issubdtype(embeddings.dtype, np.number):
            raise TypeError(f"embeddings are not numeric: {embeddings.dtype}")
        row_count = embeddings.shape[0]
        if not (len(image_ids) == len(stored_paths) == len(identities) == row_count):
            raise ValueError("embedding, identifier, path, and label lengths differ")
        if len(np.unique(image_ids)) != row_count:
            raise ValueError("duplicate image/sample identifiers in cache")
        if [str(value) for value in np.asarray(dataset_values).ravel().tolist()] != [dataset_name]:
            raise ValueError(f"dataset array mismatch: {dataset_values.tolist()}")

        expected_ids = expected_frame["image_id"].astype(str).tolist()
        expected_labels = expected_frame["identity"].astype(str).tolist()
        expected_paths = expected_frame["path"].astype(str).tolist()
        if [str(value) for value in np.asarray(image_ids).ravel().tolist()] != expected_ids:
            raise ValueError("image identifiers/order do not match the frozen manifest")
        if [str(value) for value in np.asarray(identities).ravel().tolist()] != expected_labels:
            raise ValueError("identity labels/order do not match the frozen manifest")
        if [str(value) for value in np.asarray(stored_paths).ravel().tolist()] != expected_paths:
            raise ValueError("stored paths/order do not match the frozen manifest")

        if metadata.get("model") != model_name or metadata.get("dataset") != dataset_name:
            raise ValueError("model/dataset provenance mismatch")
        if int(metadata.get("number_images", -1)) != row_count:
            raise ValueError("number_images provenance mismatch")
        if int(metadata.get("embedding_dimension", -1)) != embeddings.shape[1]:
            raise ValueError("embedding_dimension provenance mismatch")
        if metadata.get("dataset_manifest_sha256") != DATASET_MANIFEST_HASHES[dataset_name]:
            raise ValueError("logical dataset manifest hash mismatch")
        expected_order_hash = sha256_bytes(
            "\n".join(str(value) for value in np.asarray(image_ids).ravel().tolist()).encode("utf-8")
        )
        if metadata.get("image_order_sha256") != expected_order_hash:
            raise ValueError("image_order_sha256 mismatch")

        norm_min = float("inf")
        norm_max = float("-inf")
        max_norm_error = 0.0
        for start in range(0, row_count, 8192):
            block = embeddings[start:start + 8192]
            if not np.isfinite(block).all():
                raise ValueError("embedding cache contains NaN or Inf")
            norms = np.linalg.norm(block, axis=1)
            if np.any(norms <= 1e-12):
                raise ValueError("embedding cache contains zero-norm rows")
            norm_min = min(norm_min, float(norms.min()))
            norm_max = max(norm_max, float(norms.max()))
            max_norm_error = max(max_norm_error, float(np.max(np.abs(norms - 1.0))))
        declared_normalized = metadata.get("l2_normalized") is True
        measured_normalized = max_norm_error <= L2_NORMALIZATION_ATOL
        if declared_normalized and not measured_normalized:
            raise ValueError(
                f"cache declares L2 normalization but max |norm-1|={max_norm_error:.6g}"
            )
        if not declared_normalized:
            raise ValueError("baseline evaluator requires cache metadata l2_normalized=true")

        file_sha256 = sha256_file(path)
        manifest_sha256 = sha256_file(manifest_paths[dataset_name])
        status = "OK"
        result = {
            "model": model_name,
            "dataset": dataset_name,
            "embedding_file": str(path.resolve()),
            "source_embedding_cache": str(path.resolve()),
            "num_samples": int(row_count),
            "embedding_dim": int(embeddings.shape[1]),
            "dtype": str(embeddings.dtype),
            "l2_normalized": bool(measured_normalized),
            "l2_norm_min": norm_min,
            "l2_norm_max": norm_max,
            "l2_max_abs_error": max_norm_error,
            "sha256": file_sha256,
            "logical_dataset_hash": DATASET_MANIFEST_HASHES[dataset_name],
            "manifest_file": str(manifest_paths[dataset_name].resolve()),
            "manifest_sha256": manifest_sha256,
            "model_identifier": metadata.get("model_identifier", "unknown") or "unknown",
            "model_revision": metadata.get("model_revision", "unknown") or "unknown",
            "checkpoint_sha256": metadata.get("checkpoint_sha256", "unknown") or "unknown",
            "preprocessing": stable_json(metadata.get("preprocessing_signature", "unknown")),
            "normalization_convention": "stored L2-normalized; explicitly normalized again in float32 scoring blocks",
            "source_file_mtime_utc": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat(),
            "original_provenance_metadata": stable_json(metadata),
            "status": status,
        }
        return result
    except Exception as exc:
        raise RuntimeError(
            f"Invalid embedding cache:\n\nModel: {model_name}\nDataset: {dataset_name}\n"
            f"Path: {path}\nReason: {type(exc).__name__}: {exc}\n\n"
            "If this is an object-array pickle error, re-run the helpers cell then this "
            "validation cell. Do not set FORCE_REEXTRACT=True; embeddings are already on disk."
        ) from exc
    finally:
        record_runtime("embedding_cache_validation", started, model=model_name, dataset=dataset_name)


inventory_rows = []
cache_headers = {}
found_models = set()
validated_dimensions = {}
for model_index, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"[{model_index}/{len(MODEL_NAMES)}] {model_name} — validating six frozen caches")
    for dataset_name in REQUIRED_DATASETS:
        row = validate_embedding_cache(model_name, dataset_name, canonical_frames[dataset_name])
        validated_dimensions.setdefault(model_name, row["embedding_dim"])
        if validated_dimensions[model_name] != row["embedding_dim"]:
            raise RuntimeError(f"{model_name}: inconsistent validated embedding dimension")
        inventory_rows.append(row)
        cache_headers[(model_name, dataset_name)] = inspect_npz_header(cache_path(model_name, dataset_name))
        found_models.add(model_name)
        print(
            f"    {dataset_name:<20} {row['num_samples']:>7,} x {row['embedding_dim']:<5} "
            f"norm_err={row['l2_max_abs_error']:.2e} OK"
        )
        gc.collect()

embedding_inventory = pd.DataFrame(inventory_rows)
if len(embedding_inventory) != len(MODEL_NAMES) * len(REQUIRED_DATASETS):
    raise RuntimeError("Embedding inventory is incomplete")
if set(embedding_inventory["model"]) != EXPECTED_MODELS:
    raise RuntimeError("Active embedding inventory does not contain exactly the requested BHCA models")
atomic_write_csv(OUTPUT_ROOT / "embedding_inventory.csv", embedding_inventory)
atomic_write_csv(OUTPUT_ROOT / "tables" / "embedding_cache_index.csv", embedding_inventory)
print("RIGOROUS EMBEDDING VALIDATION: PASSED")
print("Inventory:", OUTPUT_ROOT / "embedding_inventory.csv")


## 5. Rejection and clustering calibration algorithms

Both supported rejection modes use the same L2-normalized identity-centroid cosine scorer and the same identity-balanced `NormAcc` threshold rule. `raw_max_cosine` applies that rule to full-gallery `s1` as the preserved ablation.

The official `margin_gallery_logistic` mode builds deterministic pseudo-open-set episodes from validation data only. Gallery sizes follow a generic geometric schedule, every episode has at least two identities, and the full calibration gallery is always included. Query-to-full-gallery similarities are computed once per bounded query block and sliced for each episode, reusing centroids without constructing an all-query similarity matrix.

The exact fitting weight for row `i` is initially `1 / n(identity_i)`. Each class total is then rescaled to `1/2`, and all weights are finally divided by their mean. Thus known and unknown contribute equally, every identity contributes equally within its class, and the stable mean weight is one.

Standardization and logistic fitting are repeated inside identity-disjoint OOF folds. The frozen probability threshold is selected only from held-out `P(known)` values. The final standardizer and logistic coefficients are then refit on all validation episode rows. Clustering remains the existing validation-unknown average-linkage calibration and is not modified by the rejection protocol.


In [5]:
# ============================================================
# 5. Baseline scoring, rejection selection, and clustering
# ============================================================

def load_calibration_cache(model_name: str):
    path = cache_path(model_name, "apexid_calibration")
    rewrite_npz_object_strings(path)
    with np.load(path, allow_pickle=False) as cached:
        embeddings = cached["embeddings"]
        image_ids = cached["image_ids"].astype(str)
        identities = cached["identities"].astype(str)
        metadata = json.loads(str(cached["metadata_json"].item()))
    expected = canonical_frames["apexid_calibration"]
    if not same_str_array(image_ids, expected["image_id"].astype(str).tolist()):
        raise RuntimeError(f"{model_name}: calibration cache image order changed after preflight")
    if not same_str_array(identities, expected["identity"].astype(str).tolist()):
        raise RuntimeError(f"{model_name}: calibration cache labels changed after preflight")
    return embeddings, metadata


def build_identity_gallery_stable(embeddings: np.ndarray, identities):
    values = normalize_for_cosine(embeddings, context="train gallery")
    labels = np.asarray(identities).astype(str)
    if len(labels) != len(values) or len(labels) == 0:
        raise ValueError("Gallery labels/embeddings are empty or misaligned")
    order = np.argsort(labels, kind="stable")
    sorted_labels = labels[order]
    sorted_values = values[order]
    starts = np.r_[0, np.flatnonzero(sorted_labels[1:] != sorted_labels[:-1]) + 1]
    stops = np.r_[starts[1:], len(sorted_labels)]
    unique = sorted_labels[starts]
    centroids = np.vstack([
        sorted_values[start:stop].mean(axis=0)
        for start, stop in zip(starts, stops)
    ]).astype(np.float32, copy=False)
    centroids = normalize_for_cosine(centroids, context="identity centroids")
    counts = (stops - starts).astype(np.int64)
    return unique.astype(str), centroids, counts


def score_top2_identity_gallery(query_embeddings: np.ndarray, gallery_identities: np.ndarray, centroids: np.ndarray):
    if len(gallery_identities) < 2:
        raise ValueError("At least two gallery identities are required for s1/s2 diagnostics")
    number_queries = len(query_embeddings)
    s1 = np.empty(number_queries, dtype=np.float32)
    s2 = np.empty(number_queries, dtype=np.float32)
    best_indices = np.empty(number_queries, dtype=np.int64)
    for start in range(0, number_queries, SIMILARITY_CHUNK_SIZE):
        stop = min(start + SIMILARITY_CHUNK_SIZE, number_queries)
        query = normalize_for_cosine(query_embeddings[start:stop], context="validation queries")
        similarities = query @ centroids.T
        rows = np.arange(stop - start)
        first = np.argmax(similarities, axis=1)
        s1[start:stop] = similarities[rows, first]
        best_indices[start:stop] = first
        similarities[rows, first] = -np.inf
        second = np.argmax(similarities, axis=1)
        s2[start:stop] = similarities[rows, second]
    return s1, s2, gallery_identities[best_indices]


def balanced_mean(labels, values) -> float:
    labels = np.asarray(labels).astype(str)
    values = np.asarray(values, dtype=float)
    unique, inverse = np.unique(labels, return_inverse=True)
    if len(unique) == 0:
        return float("nan")
    sums = np.bincount(inverse, weights=values, minlength=len(unique))
    counts = np.bincount(inverse, minlength=len(unique))
    return float(np.mean(sums / counts))


def norm_accuracy(baks: float, baus: float) -> float:
    return float(math.sqrt(max(float(baks), 0.0) * max(float(baus), 0.0)))


def select_threshold_by_normacc(scores, true_known_mask, true_labels, predicted_labels) -> float:
    """Exact baseline sweep: highest threshold among maxima of identity-balanced NormAcc."""
    scores = np.asarray(scores, dtype=np.float64)
    known = np.asarray(true_known_mask, dtype=bool)
    truth = np.asarray(true_labels).astype(str)
    predicted = np.asarray(predicted_labels).astype(str)
    if not (len(scores) == len(known) == len(truth) == len(predicted)):
        raise ValueError("Rejection calibration arrays have different lengths")
    if not np.isfinite(scores).all() or not known.any() or known.all():
        raise ValueError("Rejection calibration needs finite scores and both known/unknown queries")

    known_positions = np.flatnonzero(known)
    unknown_positions = np.flatnonzero(~known)
    _, known_inverse, known_counts = np.unique(truth[known_positions], return_inverse=True, return_counts=True)
    _, unknown_inverse, unknown_counts = np.unique(truth[unknown_positions], return_inverse=True, return_counts=True)
    known_counts = known_counts.astype(np.float64)
    unknown_counts = unknown_counts.astype(np.float64)
    known_inverse_by_position = np.full(len(scores), -1, dtype=np.int64)
    unknown_inverse_by_position = np.full(len(scores), -1, dtype=np.int64)
    known_inverse_by_position[known_positions] = known_inverse
    unknown_inverse_by_position[unknown_positions] = unknown_inverse
    known_correct_accepted = np.zeros(len(known_counts), dtype=np.float64)
    unknown_accepted = np.zeros(len(unknown_counts), dtype=np.float64)

    def current_score():
        baks = float(np.mean(known_correct_accepted / known_counts))
        baus = float(np.mean((unknown_counts - unknown_accepted) / unknown_counts))
        return norm_accuracy(baks, baus)

    best_threshold = float(np.nextafter(np.max(scores), np.inf))
    best_score = current_score()
    order = np.argsort(-scores, kind="stable")
    cursor = 0
    while cursor < len(order):
        threshold = scores[order[cursor]]
        next_cursor = cursor + 1
        while next_cursor < len(order) and scores[order[next_cursor]] == threshold:
            next_cursor += 1
        indices = order[cursor:next_cursor]
        known_group = indices[known[indices]]
        correct_group = known_group[predicted[known_group] == truth[known_group]]
        if len(correct_group):
            np.add.at(known_correct_accepted, known_inverse_by_position[correct_group], 1.0)
        unknown_group = indices[~known[indices]]
        if len(unknown_group):
            np.add.at(unknown_accepted, unknown_inverse_by_position[unknown_group], 1.0)
        value = current_score()
        if value > best_score or (value == best_score and float(threshold) > best_threshold):
            best_score = value
            best_threshold = float(threshold)
        cursor = next_cursor
    return best_threshold


def rejection_metrics(scores, predicted, truth, is_novel, threshold) -> dict:
    scores = np.asarray(scores, dtype=float)
    predicted = np.asarray(predicted).astype(str)
    truth = np.asarray(truth).astype(str)
    novel = np.asarray(is_novel, dtype=bool)
    known = ~novel
    accepted = scores >= float(threshold)
    baks = balanced_mean(truth[known], (accepted[known] & (predicted[known] == truth[known])).astype(float))
    baus = balanced_mean(truth[novel], (~accepted[novel]).astype(float))
    return {"BAKS": baks, "BAUS": baus, "NormAcc": norm_accuracy(baks, baus)}


def rejection_curve(scores, predicted, truth, is_novel, selected_threshold) -> pd.DataFrame:
    scores = np.asarray(scores, dtype=float)
    all_candidates = np.unique(np.r_[np.nextafter(scores.min(), -np.inf), scores, np.nextafter(scores.max(), np.inf)])
    if len(all_candidates) > 501:
        positions = np.unique(np.rint(np.linspace(0, len(all_candidates) - 1, 501)).astype(int))
        candidates = all_candidates[positions]
    else:
        candidates = all_candidates
    candidates = np.unique(np.r_[candidates, selected_threshold])
    rows = []
    novel = np.asarray(is_novel, dtype=bool)
    truth = np.asarray(truth).astype(str)
    predicted = np.asarray(predicted).astype(str)
    for value in candidates:
        metrics = rejection_metrics(scores, predicted, truth, novel, value)
        rows.append({
            "rejection_threshold": float(value),
            **metrics,
            "selected": bool(value == selected_threshold),
        })
    return pd.DataFrame(rows)


def b3_metrics(true_labels, predicted_labels) -> dict:
    true = np.asarray(true_labels).astype(str)
    predicted = np.asarray(predicted_labels)
    contingency = Counter(zip(true.tolist(), predicted.tolist()))
    true_sizes = Counter(true.tolist())
    predicted_sizes = Counter(predicted.tolist())
    precision = sum(count * count / predicted_sizes[p] for (_t, p), count in contingency.items()) / len(true)
    recall = sum(count * count / true_sizes[t] for (t, _p), count in contingency.items()) / len(true)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"b3_precision": float(precision), "b3_recall": float(recall), "b3_f1": float(f1)}


def discovery_metrics(true_labels, predicted_labels) -> dict:
    true = np.asarray(true_labels).astype(str)
    predicted = np.asarray(predicted_labels)
    base = {
        "ari": float(adjusted_rand_score(true, predicted)),
        "ami": float(adjusted_mutual_info_score(true, predicted)),
        "nmi": float(normalized_mutual_info_score(true, predicted)),
        **b3_metrics(true, predicted),
    }
    true_groups = defaultdict(set)
    predicted_groups = defaultdict(set)
    for target, guess in zip(true, predicted):
        true_groups[target].add(int(guess))
        predicted_groups[int(guess)].add(target)
    fragments = np.asarray([len(value) for value in true_groups.values()], dtype=float)
    merges = np.asarray([len(value) for value in predicted_groups.values()], dtype=float)
    base.update({
        "mean_fragments_per_identity": float(fragments.mean()),
        "median_fragments_per_identity": float(np.median(fragments)),
        "max_fragments_per_identity": float(fragments.max()),
        "fraction_true_identities_fragmented": float(np.mean(fragments > 1)),
        "mean_true_ids_per_predicted_cluster": float(merges.mean()),
        "max_true_ids_per_predicted_cluster": float(merges.max()),
        "fraction_predicted_clusters_merged": float(np.mean(merges > 1)),
        "true_identity_count": float(len(true_groups)),
        "predicted_cluster_count": float(len(predicted_groups)),
        "absolute_cluster_count_error": float(abs(len(predicted_groups) - len(true_groups))),
        "relative_cluster_count_error": float(abs(len(predicted_groups) - len(true_groups)) / len(true_groups)),
    })
    return base


def hierarchy_metadata(model_name, unknown_image_ids, embedding_metadata) -> dict:
    return {
        "protocol_version": CALIBRATION_PROTOCOL_VERSION,
        "model": model_name,
        "linkage": "average",
        "metric": "cosine",
        "number_samples": int(len(unknown_image_ids)),
        "unknown_image_order_sha256": sha256_bytes("\n".join(map(str, unknown_image_ids)).encode("utf-8")),
        "embedding_provenance_sha256": sha256_bytes(stable_json(embedding_metadata).encode("utf-8")),
    }


def load_or_build_hierarchy(model_name, unknown_embeddings, unknown_image_ids, embedding_metadata):
    expected = hierarchy_metadata(model_name, unknown_image_ids, embedding_metadata)
    output_path = OUTPUT_ROOT / "calibration" / "hierarchies" / model_name / "val_unknown_average_cosine.npz"
    source_path = EXISTING_STAGE01_ROOT / "calibration" / "hierarchies" / model_name / "val_unknown_average_cosine.npz"
    for candidate in (output_path, source_path):
        if not candidate.is_file():
            continue
        try:
            rewrite_npz_object_strings(candidate)
            with np.load(candidate, allow_pickle=False) as cached:
                observed = json.loads(str(cached["metadata_json"].item()))
                matrix = cached["linkage_matrix"].astype(np.float64)
                number_samples = int(cached["number_samples"].item())
            for key in (
                "model", "linkage", "metric", "number_samples",
                "unknown_image_order_sha256", "embedding_provenance_sha256",
            ):
                if observed.get(key) != expected.get(key):
                    raise ValueError(f"provenance mismatch for {key}")
            expected_shape = (max(number_samples - 1, 0), 4)
            if matrix.shape != expected_shape or not np.isfinite(matrix).all():
                raise ValueError(f"invalid linkage matrix shape/content: {matrix.shape}")
            if candidate != output_path:
                atomic_save_npz(
                    output_path,
                    metadata_json=np.asarray(stable_json(expected)),
                    number_samples=np.asarray(number_samples, dtype=np.int64),
                    linkage_matrix=matrix,
                )
            print(f"    Reused validated clustering hierarchy: {candidate}")
            return matrix, True
        except Exception as exc:
            print(f"    Hierarchy cache rejected ({candidate}): {exc}")

    values = normalize_for_cosine(unknown_embeddings, context="validation-unknown clustering")
    if len(values) <= 1:
        matrix = np.empty((0, 4), dtype=np.float64)
    else:
        distances = pdist(values, metric="cosine")
        matrix = linkage(distances, method="average", optimal_ordering=False)
        del distances
    atomic_save_npz(
        output_path,
        metadata_json=np.asarray(stable_json(expected)),
        number_samples=np.asarray(len(values), dtype=np.int64),
        linkage_matrix=np.asarray(matrix, dtype=np.float64),
    )
    return matrix, False


def calibrate_clustering(linkage_matrix, number_samples: int, true_labels):
    rows = []
    for tau in sorted(set(map(float, CLUSTERING_THRESHOLD_CANDIDATES))):
        if number_samples == 1:
            predicted = np.zeros(1, dtype=np.int64)
        else:
            predicted = fcluster(linkage_matrix, t=1.0 - tau, criterion="distance").astype(np.int64) - 1
        rows.append({"clustering_threshold": tau, **discovery_metrics(true_labels, predicted)})
    curve = pd.DataFrame(rows)
    best_value = float(curve["b3_f1"].max())
    threshold = float(curve.loc[curve["b3_f1"].eq(best_value), "clustering_threshold"].max())
    curve["selected"] = curve["clustering_threshold"].eq(threshold)
    return threshold, curve




def stable_seed(*parts) -> int:
    digest = hashlib.sha256("::".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big", signed=False) % (2**32 - 1)


def build_gallery_size_schedule(number_identities: int) -> list[int]:
    number_identities = int(number_identities)
    if number_identities < 2:
        raise ValueError("At least two calibration gallery identities are required")
    if number_identities <= 8:
        count = min(4, number_identities - 1)
        values = np.rint(np.geomspace(2, number_identities, num=max(count, 2))).astype(int)
    else:
        values = []
        value = 8
        while value <= number_identities:
            values.append(value)
            value *= 2
    return sorted({int(value) for value in [*values, number_identities] if 2 <= int(value) <= number_identities})


def calibration_episode_definitions(gallery_identities, *, episodes_per_size, seed):
    gallery_identities = np.asarray(gallery_identities).astype(str)
    sizes = build_gallery_size_schedule(len(gallery_identities))
    definitions = []
    for gallery_size in sizes:
        for repeat in range(int(episodes_per_size)):
            if gallery_size == len(gallery_identities):
                indices = np.arange(len(gallery_identities), dtype=np.int64)
            else:
                generator = np.random.default_rng(stable_seed(seed, gallery_size, repeat))
                indices = np.sort(generator.choice(len(gallery_identities), size=gallery_size, replace=False))
            selected = gallery_identities[indices]
            definitions.append({
                "episode_id": f"M{gallery_size:06d}_R{repeat:02d}",
                "gallery_size": int(gallery_size),
                "gallery_indices": indices,
                "gallery_identity_sha256": sha256_bytes("\n".join(selected).encode("utf-8")),
            })
    return sizes, definitions


def generate_logistic_episode_table(
    *, model_name, validation_embeddings, validation_frame, gallery_identities,
    gallery_centroids, episodes_per_size=EPISODES_PER_GALLERY_SIZE,
    seed=CALIBRATION_EPISODE_SEED,
):
    if set(validation_frame["calibration_role"].astype(str)) - {"val_known", "val_unknown"}:
        raise RuntimeError("Only frozen validation rows may enter logistic episodes")
    labels = validation_frame["identity"].astype(str).to_numpy()
    image_ids = validation_frame["image_id"].astype(str).to_numpy()
    roles = validation_frame["calibration_role"].astype(str).to_numpy()
    is_unknown = roles == "val_unknown"
    species = (
        validation_frame["species"].astype(str).to_numpy()
        if "species" in validation_frame else np.full(len(validation_frame), "unknown", dtype=str)
    )
    full_gallery = set(np.asarray(gallery_identities).astype(str))
    if set(labels[is_unknown]) & full_gallery:
        raise RuntimeError("A val_unknown identity overlaps the complete calibration gallery")
    query = normalize_for_cosine(validation_embeddings, context="logistic calibration queries")
    gallery_sizes, definitions = calibration_episode_definitions(
        gallery_identities, episodes_per_size=episodes_per_size, seed=seed,
    )
    frames = []
    for start in range(0, len(query), SIMILARITY_CHUNK_SIZE):
        stop = min(start + SIMILARITY_CHUNK_SIZE, len(query))
        similarities = query[start:stop] @ gallery_centroids.T
        block_labels = labels[start:stop]
        block_unknown = is_unknown[start:stop]
        selection_cache = {}
        for definition in definitions:
            indices = definition["gallery_indices"]
            cache_key = definition["gallery_identity_sha256"]
            if cache_key not in selection_cache:
                selected_scores = similarities[:, indices].copy()
                rows = np.arange(len(selected_scores))
                first = np.argmax(selected_scores, axis=1)
                top1 = selected_scores[rows, first].astype(np.float32)
                predicted = np.asarray(gallery_identities).astype(str)[indices[first]]
                selected_scores[rows, first] = -np.inf
                top2 = np.max(selected_scores, axis=1).astype(np.float32)
                selection_cache[cache_key] = (top1, top2, predicted)
            top1, top2, predicted = selection_cache[cache_key]
            selected_labels = np.asarray(gallery_identities).astype(str)[indices]
            eligible = block_unknown | np.isin(block_labels, selected_labels)
            if not eligible.any():
                continue
            positions = np.flatnonzero(eligible)
            absolute = start + positions
            size = int(definition["gallery_size"])
            frames.append(pd.DataFrame({
                "model": model_name,
                "episode_id": definition["episode_id"],
                "gallery_identity_sha256": definition["gallery_identity_sha256"],
                "gallery_size": size,
                "log_gallery_size": math.log(float(size)),
                "image_id": image_ids[absolute],
                "true_identity": labels[absolute],
                "is_known": ~is_unknown[absolute],
                "is_unknown": is_unknown[absolute],
                "predicted_nearest_identity": predicted[positions],
                "s1": top1[positions],
                "s2": top2[positions],
                "margin": top1[positions] - top2[positions],
                "species": species[absolute],
                "calibration_role": roles[absolute],
                "contains_final_test_label": False,
            }))
    if not frames:
        raise RuntimeError("No logistic calibration episode rows were generated")
    episode_frame = pd.concat(frames, ignore_index=True)
    episode_frame.insert(0, "row_id", np.arange(len(episode_frame), dtype=np.int64))
    definition_by_id = {item["episode_id"]: item for item in definitions}
    for episode_id, known_rows in episode_frame.loc[episode_frame["is_known"]].groupby("episode_id"):
        selected = set(np.asarray(gallery_identities).astype(str)[definition_by_id[episode_id]["gallery_indices"]])
        if not set(known_rows["true_identity"].astype(str)).issubset(selected):
            raise RuntimeError(f"{episode_id}: a known query identity is absent from its episode gallery")
    if episode_frame["contains_final_test_label"].astype(bool).any():
        raise RuntimeError("Final-test rows entered logistic calibration")
    if not (episode_frame["gallery_size"].astype(int) >= 2).all():
        raise RuntimeError("A logistic episode has fewer than two gallery identities")
    if not np.isfinite(episode_frame["log_gallery_size"].to_numpy(dtype=float)).all():
        raise RuntimeError("Non-finite log gallery size")
    if not (episode_frame["s1"].to_numpy() + 1e-6 >= episode_frame["s2"].to_numpy()).all():
        raise RuntimeError("Top-2 ordering failed")
    if not np.allclose(
        episode_frame["margin"].to_numpy(dtype=float),
        episode_frame["s1"].to_numpy(dtype=float) - episode_frame["s2"].to_numpy(dtype=float),
        rtol=0.0, atol=2e-7,
    ):
        raise RuntimeError("Episode margin is inconsistent with s1 - s2")
    return episode_frame, gallery_sizes, definitions


IDENTITY_BALANCED_WEIGHTING_DEFINITION = (
    "base_i=1/N_rows(true_identity_i); rescale known and unknown base-weight totals "
    "independently to 0.5; divide all weights by their global mean so mean(weight)=1"
)


def identity_balanced_fit_weights(true_identities, is_known):
    labels = np.asarray(true_identities).astype(str)
    target = np.asarray(is_known, dtype=bool)
    if not target.any() or target.all():
        raise ValueError("Identity-balanced fitting requires known and unknown rows")
    weights = np.empty(len(labels), dtype=np.float64)
    for class_value in (False, True):
        positions = np.flatnonzero(target == class_value)
        unique, inverse, counts = np.unique(labels[positions], return_inverse=True, return_counts=True)
        if not len(unique):
            raise ValueError("A fitting class has no identities")
        class_weights = 1.0 / counts[inverse].astype(np.float64)
        class_weights *= 0.5 / class_weights.sum()
        weights[positions] = class_weights
    weights /= weights.mean()
    if not np.isfinite(weights).all() or np.any(weights <= 0):
        raise RuntimeError("Invalid identity-balanced fit weights")
    return weights


def fit_weighted_standardizer(values, sample_weight):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(sample_weight, dtype=np.float64)
    mean = np.average(values, axis=0, weights=weights)
    variance = np.average((values - mean) ** 2, axis=0, weights=weights)
    scale = np.sqrt(variance)
    if not np.isfinite(mean).all() or not np.isfinite(scale).all() or np.any(scale <= 1e-12):
        raise RuntimeError("Feature standardizer has a non-finite or zero scale")
    return mean, scale


def deterministic_grouped_folds(target, groups, n_splits, seed):
    target = np.asarray(target, dtype=int)
    groups = np.asarray(groups).astype(str)
    if StratifiedGroupKFold is not None:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=int(seed))
        yield from splitter.split(np.zeros((len(target), 1)), target, groups)
        return
    generator = np.random.default_rng(int(seed))
    group_fold = {}
    for class_value in (0, 1):
        class_groups = np.unique(groups[target == class_value])
        generator.shuffle(class_groups)
        for index, group in enumerate(class_groups):
            group_fold[group] = index % n_splits
    for fold in range(n_splits):
        held = np.flatnonzero(np.asarray([group_fold[group] == fold for group in groups]))
        train = np.flatnonzero(np.asarray([group_fold[group] != fold for group in groups]))
        yield train, held


def stable_sigmoid(values):
    values = np.asarray(values, dtype=np.float64)
    output = np.empty_like(values)
    positive = values >= 0
    output[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
    exp_values = np.exp(values[~positive])
    output[~positive] = exp_values / (1.0 + exp_values)
    return output


def apply_portable_logistic(features, calibrator):
    values = np.asarray(features, dtype=np.float64)
    if tuple(calibrator["feature_names"]) != LOGISTIC_FEATURE_NAMES:
        raise ValueError("Portable calibrator feature order mismatch")
    mean = np.asarray(calibrator["standardizer"]["mean"], dtype=np.float64)
    scale = np.asarray(calibrator["standardizer"]["scale"], dtype=np.float64)
    coef = np.asarray(calibrator["logistic_regression"]["coef"], dtype=np.float64).reshape(-1)
    intercept = float(calibrator["logistic_regression"]["intercept"])
    if values.shape[1] != len(mean) or not (len(mean) == len(scale) == len(coef)):
        raise ValueError("Portable calibrator dimensions are inconsistent")
    probabilities = stable_sigmoid(((values - mean) / scale) @ coef + intercept)
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise RuntimeError("Portable calibrator produced invalid probabilities")
    return probabilities


def fit_grouped_oof_logistic(episode_frame, *, seed=CALIBRATION_EPISODE_SEED):
    features = episode_frame.loc[:, LOGISTIC_FEATURE_NAMES].to_numpy(dtype=np.float64)
    target = episode_frame["is_known"].to_numpy(dtype=int)
    groups = episode_frame["true_identity"].astype(str).to_numpy()
    number_known = len(np.unique(groups[target == 1]))
    number_unknown = len(np.unique(groups[target == 0]))
    n_splits = min(5, number_known, number_unknown)
    if n_splits < 3:
        raise RuntimeError("Grouped OOF logistic calibration requires at least three folds")
    oof = np.full(len(episode_frame), np.nan, dtype=np.float64)
    assignments = np.full(len(episode_frame), -1, dtype=np.int16)
    prediction_count = np.zeros(len(episode_frame), dtype=np.int8)
    fold_rows = []
    for fold, (train, held) in enumerate(deterministic_grouped_folds(target, groups, n_splits, seed)):
        train_groups = set(groups[train])
        held_groups = set(groups[held])
        overlap = train_groups & held_groups
        if overlap:
            raise RuntimeError(f"OOF fold {fold} leaks identities: {sorted(overlap)[:3]}")
        if len(np.unique(target[train])) != 2 or len(np.unique(target[held])) != 2:
            raise RuntimeError(f"OOF fold {fold} does not contain both classes")
        weights = identity_balanced_fit_weights(groups[train], target[train].astype(bool))
        mean, scale = fit_weighted_standardizer(features[train], weights)
        estimator = LogisticRegression(
            C=LOGISTIC_C, penalty="l2", solver=LOGISTIC_SOLVER,
            max_iter=LOGISTIC_MAX_ITER, random_state=int(seed),
        )
        estimator.fit((features[train] - mean) / scale, target[train], sample_weight=weights)
        probabilities = estimator.predict_proba((features[held] - mean) / scale)[:, 1]
        if not np.isfinite(probabilities).all():
            raise RuntimeError(f"OOF fold {fold} produced non-finite probabilities")
        oof[held] = probabilities
        assignments[held] = fold
        prediction_count[held] += 1
        fold_rows.append({
            "fold": fold,
            "train_rows": len(train),
            "held_out_rows": len(held),
            "train_identities": len(train_groups),
            "held_out_identities": len(held_groups),
            "train_known_identities": len(np.unique(groups[train][target[train] == 1])),
            "train_unknown_identities": len(np.unique(groups[train][target[train] == 0])),
            "held_out_known_identities": len(np.unique(groups[held][target[held] == 1])),
            "held_out_unknown_identities": len(np.unique(groups[held][target[held] == 0])),
            "identity_overlap_count": len(overlap),
            "held_out_identity_sha256": sha256_bytes("\n".join(sorted(held_groups)).encode("utf-8")),
        })
    if np.isnan(oof).any() or not np.isfinite(oof).all() or np.any((oof < 0) | (oof > 1)):
        raise RuntimeError("OOF probabilities are missing or outside [0, 1]")
    if not np.all(prediction_count == 1) or np.any(assignments < 0):
        raise RuntimeError("Every calibration row must receive exactly one OOF prediction")
    all_weights = identity_balanced_fit_weights(groups, target.astype(bool))
    final_mean, final_scale = fit_weighted_standardizer(features, all_weights)
    final_estimator = LogisticRegression(
        C=LOGISTIC_C, penalty="l2", solver=LOGISTIC_SOLVER,
        max_iter=LOGISTIC_MAX_ITER, random_state=int(seed),
    )
    final_estimator.fit((features - final_mean) / final_scale, target, sample_weight=all_weights)
    if not np.isfinite(final_estimator.coef_).all() or not np.isfinite(final_estimator.intercept_).all():
        raise RuntimeError("Final logistic coefficients are not finite")
    return {
        "features": features,
        "target": target,
        "oof_probabilities": oof,
        "oof_fold": assignments,
        "oof_prediction_count": prediction_count,
        "fold_summary": pd.DataFrame(fold_rows),
        "n_splits": n_splits,
        "final_mean": final_mean,
        "final_scale": final_scale,
        "final_coef": final_estimator.coef_.reshape(-1).astype(np.float64),
        "final_intercept": float(final_estimator.intercept_[0]),
        "fit_weights": all_weights,
    }


def combine_csv_files(paths, destination):
    paths = [Path(path) for path in paths]
    if not paths:
        return
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("wb", dir=destination.parent, delete=False) as output:
        temporary = Path(output.name)
        for index, path in enumerate(paths):
            with path.open("rb") as source_file:
                if index:
                    source_file.readline()
                shutil.copyfileobj(source_file, output, length=8 * 1024 * 1024)
    os.replace(temporary, destination)


print("Calibration algorithms ready:", CALIBRATION_PROTOCOL_VERSION, REJECTION_CALIBRATION_METHOD)


Calibration algorithms ready: cvpr_stage01_margin_gallery_logistic_v1 margin_gallery_logistic


## 6. Three-model validation-only calibration

Each model uses the identical episode schedule, features, weighting formula, OOF protocol, fixed `C=1.0` L2 logistic regression, and threshold-selection rule. Model-specific coefficients are expected because embedding similarity distributions differ.

The full-gallery raw-cosine result is always retained for diagnostics and ablation. In official logistic mode, all reported validation rejection metrics come from identity-held-out OOF probabilities, never from the final in-sample refit. The final refit exists only to create the frozen Stage 02 artifact. Validation-unknown clustering follows the pre-existing, separate average-linkage calibration unchanged.


In [ ]:
# ============================================================
# 6. Validation-only calibration, identical for all three BHCA models
# ============================================================

threshold_rows = []
rejection_curve_frames = []
clustering_curve_frames = []
rejection_feature_frames = []
frozen_thresholds = {}
logistic_episode_paths = []
calibration_diagnostic_rows = []
calibration_comparison_rows = []
grouped_oof_fold_counts = {}

roles = calibration_frame["calibration_role"].astype(str)
gallery_mask = roles.eq("train_gallery").to_numpy()
known_mask = roles.eq("val_known").to_numpy()
unknown_mask = roles.eq("val_unknown").to_numpy()
if np.any(gallery_mask & (known_mask | unknown_mask)) or np.any(known_mask & unknown_mask):
    raise RuntimeError("Calibration role masks overlap")
if int(gallery_mask.sum() + known_mask.sum() + unknown_mask.sum()) != len(calibration_frame):
    raise RuntimeError("Calibration role masks do not cover the frozen manifest")
if set(calibration_frame.loc[unknown_mask, "identity"].astype(str)) & set(calibration_frame.loc[gallery_mask, "identity"].astype(str)):
    raise RuntimeError("Validation-unknown identities overlap the complete train gallery")

for model_index, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"\n[{model_index}/{len(MODEL_NAMES)}] {model_name}")
    model_started = time.perf_counter()
    embeddings, embedding_metadata = load_calibration_cache(model_name)

    gallery_identities, gallery_centroids, gallery_counts = build_identity_gallery_stable(
        embeddings[gallery_mask],
        calibration_frame.loc[gallery_mask, "identity"].astype(str).to_numpy(),
    )
    gallery_size = int(len(gallery_identities))
    if gallery_size != int(split_v2.get("train_identities", gallery_size)):
        raise RuntimeError(f"{model_name}: gallery identity count differs from frozen Split V2")

    validation_indices = np.r_[np.flatnonzero(known_mask), np.flatnonzero(unknown_mask)]
    validation_frame = calibration_frame.iloc[validation_indices].reset_index(drop=True)
    validation_embeddings = embeddings[validation_indices]
    validation_labels = validation_frame["identity"].astype(str).to_numpy()
    validation_is_novel = validation_frame["calibration_role"].eq("val_unknown").to_numpy()
    validation_image_ids = validation_frame["image_id"].astype(str).to_numpy()
    inventory_row = embedding_inventory.loc[
        embedding_inventory["model"].eq(model_name)
        & embedding_inventory["dataset"].eq("apexid_calibration")
    ].iloc[0]

    # Preserve the full-gallery raw-cosine score for ablation and diagnostics.
    score_path = OUTPUT_ROOT / "scores" / model_name / "apexid_calibration" / "validation_scores.npz"
    score_metadata = {
        "schema_version": 3,
        "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
        "rejection_calibration_method": REJECTION_CALIBRATION_METHOD,
        "model": model_name,
        "dataset": "apexid_calibration",
        "split": "val_known_plus_val_unknown",
        "split_manifest_sha256": split_v2["split_manifest_sha256"],
        "embedding_file_sha256": inventory_row["sha256"],
        "gallery_scoring": "L2-normalized identity centroid cosine top-2",
        "gallery_identity_count": gallery_size,
        "contains_final_test_labels": False,
    }
    score_hit = False
    if score_path.is_file():
        try:
            rewrite_npz_object_strings(score_path)
            with np.load(score_path, allow_pickle=False) as cached:
                observed = json.loads(str(cached["metadata_json"].item()))
                if stable_json(observed) != stable_json(score_metadata):
                    raise ValueError("score provenance mismatch")
                if not np.array_equal(cached["image_ids"].astype(str), validation_image_ids):
                    raise ValueError("score image order mismatch")
                s1 = cached["s1"].astype(np.float32)
                s2 = cached["s2"].astype(np.float32)
                predicted_identities = cached["predicted_identities"].astype(str)
            score_hit = True
        except Exception as exc:
            print("    Full-gallery score cache rejected:", exc)
    if not score_hit:
        score_started = time.perf_counter()
        s1, s2, predicted_identities = score_top2_identity_gallery(
            validation_embeddings, gallery_identities, gallery_centroids
        )
        record_runtime(
            "validation_similarity_computation", score_started, model=model_name,
            number_queries=len(validation_frame), gallery_identities=gallery_size, cache_hit=False,
        )
        atomic_save_npz(
            score_path,
            metadata_json=np.asarray(stable_json(score_metadata)),
            image_ids=validation_image_ids.astype("U"),
            true_identities=validation_labels.astype("U"),
            is_novel=validation_is_novel.astype(bool),
            knownness_scores=s1.astype(np.float32),
            predicted_identities=predicted_identities.astype("U"),
            s1=s1.astype(np.float32), s2=s2.astype(np.float32),
            margin=(s1 - s2).astype(np.float32),
            gallery_size=np.full(len(s1), gallery_size, dtype=np.int64),
            log_gallery_size=np.full(len(s1), math.log(float(gallery_size)), dtype=np.float64),
        )

    raw_threshold = select_threshold_by_normacc(
        s1, ~validation_is_novel, validation_labels, predicted_identities
    )
    raw_metrics = rejection_metrics(
        s1, predicted_identities, validation_labels, validation_is_novel, raw_threshold
    )
    feature_frame = pd.DataFrame({
        "model": model_name,
        "dataset": "apexid_calibration",
        "split": validation_frame["calibration_role"].astype(str),
        "episode": "full_gallery_raw_diagnostic",
        "image_id": validation_image_ids,
        "true_identity": validation_labels,
        "is_known": ~validation_is_novel,
        "is_unknown": validation_is_novel,
        "predicted_nearest_identity": predicted_identities,
        "s1": s1, "s2": s2, "margin": s1 - s2,
        "gallery_size": gallery_size,
        "log_gallery_size": math.log(float(gallery_size)),
        "species": validation_frame["species"].astype(str) if "species" in validation_frame else "unknown",
        "rejection_calibration_method": REJECTION_CALIBRATION_METHOD,
        "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
        "contains_final_test_label": False,
    })
    per_model_features = OUTPUT_ROOT / "scores" / model_name / "validation_rejection_features.csv"
    atomic_write_csv(per_model_features, feature_frame)
    rejection_feature_frames.append(feature_frame)

    calibrator_path = None
    calibrator_sha256 = None
    logistic_oof_path = None
    logistic_fold_path = None
    logistic_gallery_sizes = []
    logistic_episode_count = 0
    logistic_episode_row_count = 0
    logistic_oof_folds = 0
    if REJECTION_CALIBRATION_METHOD == "raw_max_cosine":
        rejection_score_name = "s1"
        rejection_threshold = raw_threshold
        rejection_selected = raw_metrics
        rejection_table = rejection_curve(
            s1, predicted_identities, validation_labels, validation_is_novel, rejection_threshold
        )
        rejection_table["rejection_score_name"] = rejection_score_name
    elif REJECTION_CALIBRATION_METHOD == "margin_gallery_logistic":
        episode_started = time.perf_counter()
        episode_frame, logistic_gallery_sizes, episode_definitions = generate_logistic_episode_table(
            model_name=model_name,
            validation_embeddings=validation_embeddings,
            validation_frame=validation_frame,
            gallery_identities=gallery_identities,
            gallery_centroids=gallery_centroids,
        )
        logistic_episode_count = len(episode_definitions)
        logistic_episode_row_count = len(episode_frame)
        per_model_episode_path = OUTPUT_ROOT / "scores" / model_name / "logistic_calibration_episodes.csv"
        atomic_write_csv(per_model_episode_path, episode_frame)
        logistic_episode_paths.append(per_model_episode_path)
        record_runtime(
            "logistic_episode_generation", episode_started, model=model_name,
            episode_count=logistic_episode_count, episode_rows=logistic_episode_row_count,
        )

        fit_started = time.perf_counter()
        logistic_fit = fit_grouped_oof_logistic(episode_frame)
        logistic_oof_folds = int(logistic_fit["n_splits"])
        grouped_oof_fold_counts[model_name] = logistic_oof_folds
        oof_probabilities = logistic_fit["oof_probabilities"]
        rejection_threshold = select_threshold_by_normacc(
            oof_probabilities,
            logistic_fit["target"].astype(bool),
            episode_frame["true_identity"].astype(str),
            episode_frame["predicted_nearest_identity"].astype(str),
        )
        if not 0.0 <= float(rejection_threshold) <= 1.0:
            raise RuntimeError("Frozen probability threshold is outside [0, 1]")
        rejection_selected = rejection_metrics(
            oof_probabilities,
            episode_frame["predicted_nearest_identity"].astype(str),
            episode_frame["true_identity"].astype(str),
            episode_frame["is_unknown"].astype(bool),
            rejection_threshold,
        )
        rejection_score_name = "p_known"
        rejection_table = pd.DataFrame([{
            "rejection_threshold": float(rejection_threshold),
            **rejection_selected,
            "selected": True,
            "rejection_score_name": rejection_score_name,
            "validation_probability_source": "identity-grouped OOF",
        }])

        model_calibration_dir = OUTPUT_ROOT / "calibration" / model_name
        logistic_oof_path = model_calibration_dir / "logistic_oof_predictions.csv"
        logistic_fold_path = model_calibration_dir / "logistic_fold_summary.csv"
        oof_frame = episode_frame.loc[:, [
            "row_id", "episode_id", "gallery_size", "image_id", "true_identity",
            "is_known", "is_unknown", "predicted_nearest_identity", "s1", "s2", "margin",
        ]].copy()
        oof_frame["oof_fold"] = logistic_fit["oof_fold"]
        oof_frame["oof_prediction_count"] = logistic_fit["oof_prediction_count"]
        oof_frame["p_known_oof"] = oof_probabilities
        oof_frame["identity_balanced_fit_weight_full_refit"] = logistic_fit["fit_weights"]
        atomic_write_csv(logistic_oof_path, oof_frame)
        atomic_write_csv(logistic_fold_path, logistic_fit["fold_summary"])

        calibrator = {
            "method": "margin_gallery_logistic",
            "version": "margin_gallery_logistic_v1",
            "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
            "model": model_name,
            "feature_names": list(LOGISTIC_FEATURE_NAMES),
            "standardizer": {
                "mean": logistic_fit["final_mean"].tolist(),
                "scale": logistic_fit["final_scale"].tolist(),
                "fit_scope": "all validation logistic episode rows only",
                "sample_weighted": True,
            },
            "logistic_regression": {
                "coef": logistic_fit["final_coef"].tolist(),
                "intercept": logistic_fit["final_intercept"],
                "C": LOGISTIC_C,
                "penalty": "l2",
                "solver": LOGISTIC_SOLVER,
                "max_iter": LOGISTIC_MAX_ITER,
                "random_state": CALIBRATION_EPISODE_SEED,
                "target_class_1": "known",
            },
            "rejection_probability_threshold": float(rejection_threshold),
            "threshold_selection": "identity_balanced_NormAcc_on_grouped_OOF_predictions",
            "identity_balanced_weighting": IDENTITY_BALANCED_WEIGHTING_DEFINITION,
            "oof_folds": logistic_oof_folds,
            "episode_generation": {
                "seed": CALIBRATION_EPISODE_SEED,
                "episodes_per_gallery_size": EPISODES_PER_GALLERY_SIZE,
                "gallery_sizes": list(map(int, logistic_gallery_sizes)),
                "episode_count": logistic_episode_count,
                "episode_row_count": logistic_episode_row_count,
            },
            "provenance": {
                "embedding_file_sha256": inventory_row["sha256"],
                "split_manifest_sha256": split_v2["split_manifest_sha256"],
                "episode_table_sha256": sha256_file(per_model_episode_path),
                "oof_predictions_sha256": sha256_file(logistic_oof_path),
                "fold_summary_sha256": sha256_file(logistic_fold_path),
            },
            "contains_final_test_labels": False,
        }
        calibrator_path = model_calibration_dir / "rejection_calibrator.json"
        write_json(calibrator_path, calibrator)
        calibrator_sha256 = sha256_file(calibrator_path)
        reloaded = json.loads(calibrator_path.read_text(encoding="utf-8"))
        manual_probabilities = apply_portable_logistic(logistic_fit["features"], reloaded)
        direct_probabilities = stable_sigmoid(
            ((logistic_fit["features"] - logistic_fit["final_mean"]) / logistic_fit["final_scale"])
            @ logistic_fit["final_coef"] + logistic_fit["final_intercept"]
        )
        if not np.allclose(manual_probabilities, direct_probabilities, rtol=0.0, atol=1e-12):
            raise RuntimeError("Serialized logistic artifact does not reproduce final probabilities")
        record_runtime(
            "grouped_oof_logistic_calibration", fit_started, model=model_name,
            folds=logistic_oof_folds, episode_rows=logistic_episode_row_count,
        )
        calibration_diagnostic_rows.append({
            "model": model_name,
            "method": REJECTION_CALIBRATION_METHOD,
            "validation_oof_BAKS": rejection_selected["BAKS"],
            "validation_oof_BAUS": rejection_selected["BAUS"],
            "validation_oof_NormAcc": rejection_selected["NormAcc"],
            "probability_threshold": rejection_threshold,
            "coef_s1": logistic_fit["final_coef"][0],
            "coef_margin": logistic_fit["final_coef"][1],
            "coef_log_gallery_size": logistic_fit["final_coef"][2],
            "intercept": logistic_fit["final_intercept"],
            "episode_count": logistic_episode_count,
            "episode_row_count": logistic_episode_row_count,
            "gallery_size_min": min(logistic_gallery_sizes),
            "gallery_size_max": max(logistic_gallery_sizes),
            "oof_fold_count": logistic_oof_folds,
        })
        calibration_comparison_rows.append({
            "model": model_name,
            "raw_max_cosine_validation_NormAcc": raw_metrics["NormAcc"],
            "logistic_validation_OOF_NormAcc": rejection_selected["NormAcc"],
            "delta": rejection_selected["NormAcc"] - raw_metrics["NormAcc"],
        })
        del episode_frame, oof_frame, logistic_fit, oof_probabilities
    else:
        raise AssertionError("Unreachable rejection calibration method")

    rejection_table.insert(0, "model", model_name)

    # Clustering calibration is intentionally unchanged and val_unknown-only.
    unknown_embeddings = embeddings[unknown_mask]
    unknown_labels = calibration_frame.loc[unknown_mask, "identity"].astype(str).to_numpy()
    unknown_image_ids = calibration_frame.loc[unknown_mask, "image_id"].astype(str).to_numpy()
    hierarchy_started = time.perf_counter()
    linkage_matrix, hierarchy_cache_hit = load_or_build_hierarchy(
        model_name, unknown_embeddings, unknown_image_ids, embedding_metadata
    )
    record_runtime(
        "clustering_hierarchy", hierarchy_started, model=model_name,
        number_queries=len(unknown_labels), cache_hit=hierarchy_cache_hit,
    )
    cluster_started = time.perf_counter()
    clustering_threshold, clustering_table = calibrate_clustering(
        linkage_matrix, len(unknown_labels), unknown_labels
    )
    record_runtime(
        "clustering_threshold_sweep", cluster_started,
        model=model_name, number_queries=len(unknown_labels),
    )
    clustering_table.insert(0, "model", model_name)
    cluster_selected = clustering_table.loc[clustering_table["selected"]].iloc[0]

    threshold_row = {
        "model": model_name,
        "rejection_calibration_method": REJECTION_CALIBRATION_METHOD,
        "rejection_score_name": rejection_score_name,
        "rejection_threshold": float(rejection_threshold),
        "validation_BAKS": float(rejection_selected["BAKS"]),
        "validation_BAUS": float(rejection_selected["BAUS"]),
        "validation_NormAcc": float(rejection_selected["NormAcc"]),
        "logistic_feature_names": stable_json(list(LOGISTIC_FEATURE_NAMES)) if calibrator_path else "",
        "logistic_oof_folds": logistic_oof_folds,
        "logistic_C": LOGISTIC_C if calibrator_path else np.nan,
        "logistic_episode_count": logistic_episode_count,
        "logistic_episode_row_count": logistic_episode_row_count,
        "logistic_gallery_sizes": stable_json(list(map(int, logistic_gallery_sizes))),
        "clustering_threshold": float(clustering_threshold),
        "validation_B3_Precision": float(cluster_selected["b3_precision"]),
        "validation_B3_Recall": float(cluster_selected["b3_recall"]),
        "validation_B3_F1": float(cluster_selected["b3_f1"]),
        "validation_ARI": float(cluster_selected["ari"]),
        "validation_AMI": float(cluster_selected["ami"]),
        "validation_NMI": float(cluster_selected["nmi"]),
        "rejection_threshold_source": "ApexID Split V2 validation",
        "clustering_threshold_source": "ApexID Split V2 validation",
        "rejection_validation_probability_source": "identity-grouped OOF" if calibrator_path else "full validation raw score",
        "gallery_scoring": "L2-normalized identity centroid cosine",
        "calibration_score_backend": "numpy_cpu_chunked",
        "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
        "validation_known_samples": int(known_mask.sum()),
        "validation_unknown_samples": int(unknown_mask.sum()),
        "gallery_identity_count": gallery_size,
    }

    model_calibration_dir = OUTPUT_ROOT / "calibration" / model_name
    rejection_path = model_calibration_dir / "rejection_threshold_curve.csv"
    clustering_path = model_calibration_dir / "clustering_threshold_curve.csv"
    atomic_write_csv(rejection_path, rejection_table)
    atomic_write_csv(clustering_path, clustering_table)

    frozen_entry = {
        "rejection_method": REJECTION_CALIBRATION_METHOD,
        "rejection_score_name": rejection_score_name,
        "rejection_threshold": float(rejection_threshold),
        "clustering_threshold": float(clustering_threshold),
        "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
    }
    if calibrator_path is not None:
        frozen_entry.update({
            "rejection_probability_threshold": float(rejection_threshold),
            "rejection_calibrator_path": calibrator_path.relative_to(OUTPUT_ROOT).as_posix(),
            "rejection_calibrator_sha256": calibrator_sha256,
            "logistic_feature_names": list(LOGISTIC_FEATURE_NAMES),
        })
    frozen_thresholds[model_name] = frozen_entry

    write_json(model_calibration_dir / "calibration_record.json", {
        "provenance": {
            "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
            "rejection_calibration_method": REJECTION_CALIBRATION_METHOD,
            "model": model_name,
            "embedding_file": str(cache_path(model_name, "apexid_calibration").resolve()),
            "embedding_file_sha256": inventory_row["sha256"],
            "embedding_provenance_sha256": sha256_bytes(stable_json(embedding_metadata).encode("utf-8")),
            "split_manifest_sha256": split_v2["split_manifest_sha256"],
            "calibration_roles": ["train_gallery", "val_known", "val_unknown"],
            "contains_final_test_labels": False,
            "seed": SEED,
        },
        "threshold_row": threshold_row,
        "rejection_curve_sha256": sha256_file(rejection_path),
        "clustering_curve_sha256": sha256_file(clustering_path),
        "rejection_calibrator_path": str(calibrator_path.resolve()) if calibrator_path else None,
        "rejection_calibrator_sha256": calibrator_sha256,
        "logistic_oof_predictions_path": str(logistic_oof_path.resolve()) if logistic_oof_path else None,
        "logistic_fold_summary_path": str(logistic_fold_path.resolve()) if logistic_fold_path else None,
        "score_cache_path": str(score_path.resolve()),
        "score_cache_metadata": score_metadata,
    })

    threshold_rows.append(threshold_row)
    rejection_curve_frames.append(rejection_table)
    clustering_curve_frames.append(clustering_table)
    threshold_calibration = pd.DataFrame(threshold_rows)
    rejection_curves = pd.concat(rejection_curve_frames, ignore_index=True)
    clustering_curves = pd.concat(clustering_curve_frames, ignore_index=True)
    validation_rejection_features = pd.concat(rejection_feature_frames, ignore_index=True)
    atomic_write_csv(OUTPUT_ROOT / "calibration" / "threshold_calibration.csv", threshold_calibration)
    atomic_write_csv(OUTPUT_ROOT / "calibration" / "rejection_threshold_curves.csv", rejection_curves)
    atomic_write_csv(OUTPUT_ROOT / "calibration" / "clustering_threshold_curves.csv", clustering_curves)
    atomic_write_csv(OUTPUT_ROOT / "scores" / "validation_rejection_features.csv", validation_rejection_features)
    write_json(OUTPUT_ROOT / "calibration" / "frozen_thresholds.json", frozen_thresholds)
    if calibration_diagnostic_rows:
        atomic_write_csv(OUTPUT_ROOT / "tables" / "logistic_calibration_diagnostics.csv", pd.DataFrame(calibration_diagnostic_rows))
        atomic_write_csv(OUTPUT_ROOT / "tables" / "raw_vs_logistic_validation.csv", pd.DataFrame(calibration_comparison_rows))
    record_runtime("model_calibration_total", model_started, model=model_name)
    atomic_write_csv(OUTPUT_ROOT / "logs" / "runtime_report.csv", pd.DataFrame(runtime_rows))

    del embeddings, validation_embeddings, unknown_embeddings, gallery_centroids, linkage_matrix
    gc.collect()
    print("    DONE")

if tuple(threshold_calibration["model"]) != MODEL_NAMES:
    raise RuntimeError("Threshold output order/model set is incomplete")
if set(frozen_thresholds) != EXPECTED_MODELS:
    raise RuntimeError("Frozen threshold JSON does not contain exactly the requested BHCA models")
if REJECTION_CALIBRATION_METHOD == "margin_gallery_logistic":
    if len(logistic_episode_paths) != len(MODEL_NAMES):
        raise RuntimeError("Not all models produced logistic episode artifacts")
    if any(tuple(frozen_thresholds[name]["logistic_feature_names"]) != LOGISTIC_FEATURE_NAMES for name in MODEL_NAMES):
        raise RuntimeError("The calibrators do not use identical feature definitions")
    combine_csv_files(logistic_episode_paths, OUTPUT_ROOT / "scores" / "logistic_calibration_episodes.csv")
print("\nTHREE-MODEL BHCA VALIDATION CALIBRATION: COMPLETE")


## 7. Frozen artifacts and Stage 02 compatibility

The output root is new and does not overwrite the prior raw-cosine benchmark. Each model receives a portable JSON calibrator plus grouped-OOF predictions and fold diagnostics. `frozen_thresholds.json` retains the compatibility key `rejection_threshold` while explicitly declaring whether it is an `s1` threshold or a `p_known` probability threshold.

No-copy embedding links preserve the established Stage 02 layout. The manifest records the episode schedule, feature order, weighting equation, grouped OOF fold counts, logistic configuration, calibrator hashes, and explicit no-training/no-extraction/no-test-label guarantees.


In [ ]:
# ============================================================
# 7. Export provenance, no-copy embedding views, and manifest
# ============================================================

# Root-level convenience copies requested by the local protocol; canonical
# Stage 02 locations under calibration/ remain unchanged.
atomic_write_csv(OUTPUT_ROOT / "threshold_calibration.csv", threshold_calibration)
write_json(OUTPUT_ROOT / "frozen_thresholds.json", frozen_thresholds)

link_rows = []
if CREATE_STAGE02_COMPAT_LINKS:
    for row in inventory_rows:
        source = Path(row["embedding_file"])
        destination = OUTPUT_ROOT / "embeddings" / row["model"] / row["dataset"] / "embeddings.npz"
        mode = link_cache_for_stage02(source, destination)
        link_rows.append({
            "model": row["model"],
            "dataset": row["dataset"],
            "source_path": str(source),
            "stage02_path": str(destination),
            "link_mode": mode,
        })
embedding_links = pd.DataFrame(link_rows)
atomic_write_csv(OUTPUT_ROOT / "manifests" / "embedding_path_resolution.csv", embedding_links)

model_summary_rows = []
cache_provenance = []
for model_name in MODEL_NAMES:
    model_cache_rows = embedding_inventory.loc[embedding_inventory["model"].eq(model_name)]
    calibration_row = model_cache_rows.loc[model_cache_rows["dataset"].eq("apexid_calibration")].iloc[0]
    metadata = cache_headers[(model_name, "apexid_calibration")]["metadata"]
    model_summary_rows.append({
        "model": model_name,
        "model_identifier": metadata.get("model_identifier", "unknown") or "unknown",
        "checkpoint_path": str(MODEL_SPECS[model_name]),
        "checkpoint_sha256": metadata.get("checkpoint_sha256", "unknown") or "unknown",
        "config_path": metadata.get("config_path", "unknown") or "unknown",
        "config_sha256": metadata.get("preprocessing_signature", {}).get("config_sha256", "unknown"),
        "config_provenance": "sidecar YAML next to best_backbone.pt",
        "revision": metadata.get("model_revision", "unknown") or "unknown",
        "embedding_dimension": int(calibration_row["embedding_dim"]),
        "input_resolution": stable_json(
            metadata.get("preprocessing_signature", {}).get(
                "image_size",
                metadata.get("preprocessing_signature", {}).get(
                    "resize", metadata.get("preprocessing_signature", {}).get("input_size", "unknown")
                ),
            )
        ),
        "parameters": "unknown",
        "l2_normalized": bool(calibration_row["l2_normalized"]),
        "transform": stable_json(metadata.get("preprocessing_signature", "unknown")),
        "mixed_precision_validation": stable_json({
            "source_cache_amp_enabled": metadata.get("amp_enabled", "unknown"),
            "stage01_local_calibration_uses_cuda": False,
            "extraction_device": "cuda_if_available",
        }),
        "selected_batch_size": EXTRACT_BATCH_SIZE,
        "checkpoint_provenance_type": "bhca_backbone_sidecar",
    })
    for _, cache_row in model_cache_rows.iterrows():
        cache_provenance.append({
            "model": model_name,
            "dataset": cache_row["dataset"],
            "source_embedding_cache": cache_row["embedding_file"],
            "embedding_file_sha256": cache_row["sha256"],
            "manifest_sha256": cache_row["manifest_sha256"],
            "logical_dataset_hash": cache_row["logical_dataset_hash"],
            "number_samples": int(cache_row["num_samples"]),
            "embedding_dimension": int(cache_row["embedding_dim"]),
            "generation_timestamp": "unknown",
            "source_file_mtime_utc": cache_row["source_file_mtime_utc"],
            "original_provenance_metadata": json.loads(cache_row["original_provenance_metadata"]),
        })

model_summary = pd.DataFrame(model_summary_rows)
atomic_write_csv(OUTPUT_ROOT / "tables" / "model_summary.csv", model_summary)
atomic_write_csv(OUTPUT_ROOT / "logs" / "runtime_report.csv", pd.DataFrame(runtime_rows))

link_failures = embedding_links[
    embedding_links["link_mode"].astype(str).str.startswith("unavailable")
] if len(embedding_links) else embedding_links
stage02_direct_layout_ready = bool(len(embedding_links) == len(inventory_rows) and link_failures.empty)

episode_gallery_schedule = build_gallery_size_schedule(len(gallery_ids))
calibrator_manifest = {
    model_name: {
        "path": frozen_thresholds[model_name].get("rejection_calibrator_path"),
        "sha256": frozen_thresholds[model_name].get("rejection_calibrator_sha256"),
        "probability_threshold": frozen_thresholds[model_name].get("rejection_probability_threshold"),
    }
    for model_name in MODEL_NAMES
}

stage01_manifest = {
    "stage01_version": STAGE01_VERSION,
    "calibration_protocol_version": CALIBRATION_PROTOCOL_VERSION,
    "rejection_calibration_method": REJECTION_CALIBRATION_METHOD,
    "status": "passed",
    "mode": "bhca_extract_then_calibrate",
    "embeddings_only": False,
    "models_completed": list(MODEL_NAMES),
    "excluded_models": sorted(found_models - EXPECTED_MODELS),
    "seed": SEED,
    "n_jobs": N_JOBS,
    "model_training_performed": False,
    "embedding_extraction_performed": True,
    "test_split_regeneration": False,
    "final_test_labels_used": False,
    "logistic_feature_names": list(LOGISTIC_FEATURE_NAMES),
    "episode_generation": {
        "seed": CALIBRATION_EPISODE_SEED,
        "gallery_size_schedule": episode_gallery_schedule,
        "episodes_per_gallery_size": EPISODES_PER_GALLERY_SIZE,
        "known_query_rule": "include val_known only when its identity is in the episode gallery",
        "unknown_query_rule": "identity-disjoint frozen val_unknown only",
    },
    "grouped_oof_fold_count": grouped_oof_fold_counts,
    "logistic_regression": {
        "C": LOGISTIC_C,
        "penalty": "l2",
        "solver": LOGISTIC_SOLVER,
        "max_iter": LOGISTIC_MAX_ITER,
        "identity_balanced_weighting": IDENTITY_BALANCED_WEIGHTING_DEFINITION,
    },
    "rejection_calibrators": calibrator_manifest,
    "probability_thresholds": {
        model_name: frozen_thresholds[model_name].get("rejection_probability_threshold")
        for model_name in MODEL_NAMES
    },
    "normalization_convention": {
        "stored_embeddings": "validated L2-normalized; source files unmodified",
        "score_computation": "explicit float32 L2 normalization of query blocks and identity centroids",
    },
    "leakage_guarantees": {
        "calibration_source": ["train_gallery", "val_known", "val_unknown"],
        "final_test_labels_used": False,
        "test_splits_regenerated": False,
        "model_selection_performed": False,
        "logistic_hyperparameters_tuned_on_final_test": False,
        "official_probability_threshold_selected_on_final_test": False,
    },
    "prepared_benchmark": {
        "path": str(PREPARED_ROOT.resolve()),
        "preparation_version": PREPARATION_MANIFEST.get("preparation_version", "unknown"),
        "preparation_manifest_sha256": sha256_file(preparation_manifest_path),
        "calibration_fingerprint": PREPARATION_MANIFEST["apexid_calibration_manifest"]["logical_fingerprint"],
        "split_manifest_sha256": split_v2["split_manifest_sha256"],
        "dataset_manifest_hashes": DATASET_MANIFEST_HASHES,
    },
    "embedding_source": {
        "root": str(EMBEDDINGS_ROOT.resolve()),
        "read_only_by_protocol": False,
        "cache_count": int(len(embedding_inventory)),
        "cache_total_bytes": int(sum(Path(value).stat().st_size for value in embedding_inventory["embedding_file"])),
        "stage02_compatibility_layout": str((OUTPUT_ROOT / "embeddings").resolve()),
        "stage02_direct_layout_ready": stage02_direct_layout_ready,
        "link_manifest": str((OUTPUT_ROOT / "manifests" / "embedding_path_resolution.csv").resolve()),
    },
    "models": {
        row["model"]: {
            "model_identifier": row["model_identifier"],
            "revision": row["revision"],
            "checkpoint_sha256": row["checkpoint_sha256"],
            "embedding_dimension": int(row["embedding_dimension"]),
            "preprocessing": json.loads(row["transform"]) if row["transform"] != '"unknown"' else "unknown",
        }
        for row in model_summary_rows
    },
    "embedding_caches": cache_provenance,
    "frozen_thresholds": frozen_thresholds,
    "artifacts": {
        "embedding_inventory": {
            "path": str((OUTPUT_ROOT / "embedding_inventory.csv").resolve()),
            "sha256": sha256_file(OUTPUT_ROOT / "embedding_inventory.csv"),
        },
        "threshold_calibration": {
            "path": str((OUTPUT_ROOT / "calibration" / "threshold_calibration.csv").resolve()),
            "sha256": sha256_file(OUTPUT_ROOT / "calibration" / "threshold_calibration.csv"),
        },
        "frozen_thresholds": {
            "path": str((OUTPUT_ROOT / "calibration" / "frozen_thresholds.json").resolve()),
            "sha256": sha256_file(OUTPUT_ROOT / "calibration" / "frozen_thresholds.json"),
        },
        "logistic_calibration_episodes": {
            "path": str((OUTPUT_ROOT / "scores" / "logistic_calibration_episodes.csv").resolve())
                if logistic_episode_paths else None,
            "sha256": sha256_file(OUTPUT_ROOT / "scores" / "logistic_calibration_episodes.csv")
                if logistic_episode_paths else None,
            "contains_final_test_labels": False,
        },
        "logistic_calibration_diagnostics": {
            "path": str((OUTPUT_ROOT / "tables" / "logistic_calibration_diagnostics.csv").resolve())
                if calibration_diagnostic_rows else None,
            "sha256": sha256_file(OUTPUT_ROOT / "tables" / "logistic_calibration_diagnostics.csv")
                if calibration_diagnostic_rows else None,
        },
        "validation_rejection_features": {
            "path": str((OUTPUT_ROOT / "scores" / "validation_rejection_features.csv").resolve()),
            "sha256": sha256_file(OUTPUT_ROOT / "scores" / "validation_rejection_features.csv"),
            "contains_final_test_labels": False,
        },
    },
    "environment": {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "cpu_count": CPU_COUNT,
        "available_ram_gib_at_start": AVAILABLE_RAM_GIB if AVAILABLE_RAM_GIB is not None else "unknown",
    },
}
write_json(OUTPUT_ROOT / "stage01_manifest.json", stage01_manifest)

print("\nSTAGE 01 COMPLETE")
print("Output root:", OUTPUT_ROOT)
print("Embedding caches:", EMBEDDINGS_ROOT)
print("Stage 02 direct embedding layout ready:", stage02_direct_layout_ready)
print("Frozen thresholds:")
print(json.dumps(frozen_thresholds, indent=2))


## What Stage 02 consumes

Use the Stage 01 output root written above (`/kaggle/working/apexid_cvpr_stage01_bhca_logistic` on Kaggle, or `D:\CVPR_BM\apexid_cvpr_stage01_bhca_logistic` locally) and retain the frozen Stage 00 prepared root.

The official path is:

```text
BHCA best_backbone.pt + sidecar config
  → frozen L2 embeddings
  → deterministic validation-only gallery episodes
  → s1, s2, margin, log(gallery size)
  → identity-grouped OOF logistic P(known)
  → OOF identity-balanced NormAcc probability threshold
  → final all-validation portable calibrator refit and freeze
```

Stage 02 must never fit this calibrator. It loads and hashes `calibration/<model>/rejection_calibrator.json`, applies the saved standardizer and coefficients to test `s1`, margin, and `log(gallery_size)`, and then applies the frozen validation probability threshold.
